# La Grulla Blanca — LianQi II · V05A: mutantes y estrés histórico

**Objetivo:** ampliar V04 sin perder la autoridad de T0/T1/T2 de Sapo de Ceniza y Escarabajo de Hierro.

**Distinción esencial:** el contrato de variabilidad LII congeló temporalmente todos los ejes ordinarios al T0 para ambas especies. Por tanto, **NO hay una distribución mutante LII ratificada que podamos reproducir honestamente en combate**. Este cuaderno:

1. Verifica y muestra **100.000 spawns reales bajo el contrato mutante de LianQi I**, incluido su sistema de sufijos ratificado; **no** confunde estos spawns con monstruos LII.
2. Ejecuta **212.992 combates de sensibilidad LII** con los monstruos ratificados frente a valores superiores **históricos y expresamente no canónicos**. No otorga sufijos ni habilidades mutantes a estos escenarios.
3. Comprueba los candidatos de **Palma Ardiente DOT** y **eficiencia de Agua** contra el T0 congelado y una combinación hipotética de extremos históricos, siempre con semillas comunes.
4. Registra como pendientes la **habilitación humana de variabilidad LII**, las habilidades mutantes en combate LII, las builds con injertos y las **Concordancias legales de LII**. **No declara un cierre integral**.

| Familia de pruebas | Alcance | Cantidad |
|---|---|---:|
| Spawns mutantes LI | 5 especies y configuración oficial | 100.000 spawns (no combates) |
| Sensibilidad LII | 5 raíces, 8 builds por raíz, 4 gears, 2 especies, T1/T2, 2 políticas, 5 modos | 204.800 combates |
| Contraste focal | Fuego DOT 1×3/2×2; Agua −2 Qi, contra baseline y estrés conjunto | 8.192 combates |
| **Total** | **No canónico salvo baseline ratificada** | **212.992 combates** |

Todo sucede en laboratorio. **Sin cambios en Git, main, código HTML, tablas canónicas o contratos ratificados.** Astra integrará después del cierre.


## 1. Abrir fuente fija y elegir almacenamiento

Los runners se fijan al mismo commit validado en V04. Dejé activado Google Drive para reanudar si termina la sesión. Si no querés usar Drive, cambiá `USE_DRIVE_CHECKPOINTS` a `False` antes de ejecutar.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, zipfile, py_compile
LAB=Path('/content/LII_V05A_MUTANT_SENSITIVITY')
REPO=LAB/'repo'
SOURCE_COMMIT='9a6baa41bd6fc505a755a474cd4f64a2522ba0b9'
REPO_URL='https://github.com/Shein25/La-Grulla-Blanca.git'
WORKERS=2
USE_DRIVE_CHECKPOINTS=True
LAB.mkdir(parents=True,exist_ok=True)
if USE_DRIVE_CHECKPOINTS:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT=Path('/content/drive/MyDrive/LaGrullaBlanca_LII_V05A/results')
else:
    OUT=LAB/'results'
OUT.mkdir(parents=True,exist_ok=True)
if not (REPO/'.git').is_dir():
    if REPO.exists() and any(REPO.iterdir()):raise RuntimeError('REPO_FOLDER_NOT_EMPTY')
    subprocess.run(['git','clone','--filter=blob:none','--no-checkout',REPO_URL,str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert head==SOURCE_COMMIT
print('SOURCE:',head,'\nCHECKPOINTS:',OUT,'\nWORKERS:',WORKERS)


## 2. Instalar runners verificados

Los cuatro runners están **incluidos dentro de este `.ipynb`**; no tenés que subir archivos `.py`. Sus firmas se comparan antes de usarlos.

In [ ]:
RUNNER_SOURCES = {'LAB_RUNNER.py': '"""LII T1/T2 adaptive bridge contract gate. Lab-only; fail-closed; no canonical writes."""\nfrom __future__ import annotations\n\nimport argparse\nimport collections\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nimport copy\nimport gzip\nimport hashlib\nimport importlib\nimport inspect\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport statistics\nimport sys\nimport time\nimport zipfile\n\nLAB_ID = \'LII_DEFENSIVE_T1_T2_CAUSAL_SCREEN_V01\'\nPINNED_COMMIT = \'9a6baa41bd6fc505a755a474cd4f64a2522ba0b9\'\nSTAGE = \'LianQi_II\'\nROOTS = (\'fuego\',\'metal\',\'agua\',\'tierra\',\'viento\')\nSPECIES = (\'sapo_ceniza\',\'escarabajo_hierro\')\nGEARS = (\'MANDATORY_ENTRY\',\'CARRY_OVER_LI_HIGH\',\'EXPECTED_STAGE\',\'HIGH_ROLL_STRESS\')\nARMS = (\'T0\',\'T1\',\'T2\')\nPOLICIES = (\'UNITARGET_FIRST\',\'DEFENSE_OPEN\')\nR = 8\nWORKERS_DEFAULT = 2\nMAX_ROUNDS = 100\nSEED_BASE = \'LIIBRIDGE20261007\'\nOFF = {\'fuego\':\'palma_ardiente\',\'metal\':\'destello_plata\',\'agua\':\'latigazo_marea\',\'tierra\':\'golpe_montana\',\'viento\':\'lanza_nubes\'}\nDEF = {\'fuego\':\'cuerpo_horno\',\'metal\':\'armadura_plata\',\'agua\':\'espejo_luna\',\'tierra\':\'piel_cobre\',\'viento\':\'paso_nube\'}\nFROZEN_PATHS = {\n    \'T0\':\'experimentos/balance_nuevo/t0_lii/T0_LII_FREEZE_MANIFEST_2026-10-07.json\',\n    \'T1\':\'experimentos/balance_nuevo/t1_lii/T1_LII_REACTIVE_FREEZE_2026-10-07.json\',\n    \'T2\':\'experimentos/balance_nuevo/t2_lii/T2_LII_RECOGNITION_FREEZE_2026-10-07.json\',\n}\nINPUT_BLOBS = {\n    \'experimentos/balance_nuevo/etapa19b_combat_engine.py\':\'131519da43adabff2e1ba8b4ce711889f3563f86\',\n    \'experimentos/balance_nuevo/monster_new_engine_guard.py\':\'0dd32a940ff1ba18501f957a95ae4b8cd115b4b9\',\n    \'experimentos/balance_nuevo/techniques_arc1_catalog.json\':\'073ed88fdb9d00795a10972a4a850b22505b6d48\',\n    \'experimentos/balance_nuevo/equipment_arc1_catalog.json\':\'3ac868a399118845b91da6ccd884633e7ac2b496\',\n    \'experimentos/balance_nuevo/monster_arc1_registry.json\':\'c0d2131f2cc8dbbb96babf9711c197e54fe97591\',\n    FROZEN_PATHS[\'T0\']:\'aa9b03b8141fce2828cb39e21f1e283980cb764a\',\n    FROZEN_PATHS[\'T1\']:\'542b4265133d0cf5f88065a8fa16fe7808cf2ef7\',\n    FROZEN_PATHS[\'T2\']:\'7d5f5ad079a401d33bbcbf18911c41adf7711338\',\n}\nG = {}\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1<<20),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef git_blob(data):\n    return hashlib.sha1(b\'blob \'+str(len(data)).encode()+b\'\\x00\'+data).hexdigest()\n\n\ndef save_json(path,data):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(data,ensure_ascii=False,sort_keys=True,indent=2)+\'\\n\',encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef stable_seed(*parts):\n    d=hashlib.sha256((\'|\'.join([SEED_BASE,*map(str,parts)])).encode()).digest()\n    return int.from_bytes(d[:8],\'big\')\n\n\ndef verify_sources(repo):\n    repo=Path(repo)\n    head=os.popen(\'git -C \'+__import__(\'shlex\').quote(str(repo))+\' rev-parse HEAD\').read().strip()\n    if head!=PINNED_COMMIT:raise RuntimeError(f\'PINNED_HEAD_DRIFT:{head}:{PINNED_COMMIT}\')\n    checks={}\n    for rel,want in INPUT_BLOBS.items():\n        p=repo/rel\n        if not p.is_file():raise FileNotFoundError(f\'MISSING_INPUT:{rel}\')\n        got=git_blob(p.read_bytes())\n        if got!=want:raise RuntimeError(f\'GIT_BLOB_DRIFT:{rel}:{got}:{want}\')\n        checks[rel]={\'blob_sha\':got,\'sha256\':sha256(p),\'size\':p.stat().st_size}\n    return {\'commit\':head,\'files\':checks,\'status\':\'PASS\'}\n\n\ndef load_sources(repo):\n    repo=Path(repo)\n    base=repo/\'experimentos/balance_nuevo\'\n    if str(base) not in sys.path:sys.path.insert(0,str(base))\n    eng=importlib.import_module(\'etapa19b_combat_engine\')\n    registry=json.loads((base/\'monster_arc1_registry.json\').read_text(encoding=\'utf-8\'))\n    gear=json.loads((base/\'equipment_arc1_catalog.json\').read_text(encoding=\'utf-8\'))\n    tech=json.loads((base/\'techniques_arc1_catalog.json\').read_text(encoding=\'utf-8\'))\n    freezes={k:json.loads((repo/v).read_text(encoding=\'utf-8\')) for k,v in FROZEN_PATHS.items()}\n    # Preserve LI V04B candidate root pack used in LII T0 preflight, not a productive mutation.\n    eng.ROOTS=copy.deepcopy(eng.ROOTS)\n    eng.ROOTS[\'fuego\'].update({\'direct_damage_pct\':6.0,\'crit_chance_pp\':3.0})\n    eng.ROOTS[\'metal\'].update({\'percent_penetration_pp\':15.0,\'precision\':7.0})\n    eng.ROOTS[\'agua\'].update({\'qi_cost_mult\':0.75,\'control\':10.0})\n    G.clear();G.update(eng=eng,registry=registry,gear=gear,tech=tech,freeze=freezes)\n\n\ndef builds_for_root(root):\n    o=OFF[root]; d=DEF[root]\n    res=[(\'BASE_0\',{},0)]\n    for i in range(3):res.append((f\'OFF_T1_{i}\',{o:[i]},1))\n    for j in range(3):res.append((f\'DEF_T1_{j}\',{d:[j]},1))\n    for i in range(3):\n        for j in range(3):res.append((f\'OFF_{i}_DEF_{j}\',{o:[i],d:[j]},2))\n    assert len(res)==16\n    return res\n\n\ndef selected_gear(gid):\n    g=G[\'gear\'][\'simulation_loadouts\']\n    if gid==\'CARRY_OVER_LI_HIGH\':return tuple(g[\'HIGH_ROLL_STRESS\'][\'LianQi_I\'])\n    return tuple(g[gid][STAGE])\n\n\ndef validate_contracts():\n    eng=G[\'eng\'];reg=G[\'registry\'];f=G[\'freeze\'];tech=G[\'tech\'];gear=G[\'gear\']\n    if f[\'T0\'][\'status\']!=\'T0_LII_FROZEN\':raise RuntimeError(\'T0_FREEZE_NOT_FOUND\')\n    if f[\'T1\'][\'status\']!=\'T1_LII_REACTIVE_HUMAN_RATIFIED\':raise RuntimeError(\'T1_FREEZE_NOT_FOUND\')\n    if f[\'T2\'][\'status\']!=\'T2_LII_RECOGNITION_HUMAN_RATIFIED\':raise RuntimeError(\'T2_FREEZE_NOT_FOUND\')\n    a=f[\'T1\'][\'semantics\'];c=f[\'T2\'][\'candidate\']\n    assert a[\'activation\']==\'REACTIVE\' and a[\'consumes_monster_turn\'] is False\n    assert a[\'natural_trigger\']=={\'hp_ratio_lte\':.3,\'received_hit_ratio_gte\':.2}\n    assert \'monster_survival_cd\' in a[\'cooldown_lifecycle\']\n    assert c[\'canonical_id\']==\'T2_R2_SHORT_THREAT_40_15_PRESERVE_DUE\'\n    assert c[\'memory_window\']==2 and c[\'repeated_same_category_required\']==2\n    assert c[\'count_results\']==[\'EFECTIVA\']\n    assert c[\'anticipation\']=={\'hp_ratio_lte\':.4,\'received_hit_ratio_gte\':.15}\n    assert c[\'preserve_due_canonical_technique\'] is True\n    assert c[\'consumes_monster_turn\'] is False and c[\'changes_t1_magnitude_or_cooldown\'] is False\n    assert c[\'hidden_root_or_build_inspection\'] is False and c[\'future_rng_inspection\'] is False\n    assert set(c[\'observable_categories\'])=={\n        \'PLAYER_BASIC\',\'PLAYER_UNITARGET_TECHNIQUE\',\'PLAYER_AOE_TECHNIQUE\',\'PLAYER_DEFENSIVE_TECHNIQUE\'}\n    assert gear[\'li_balance_freeze\'][\'status\']==\'FROZEN_2026-10-07\'\n    errors=eng.validate_catalog(tech)\n    if errors:raise RuntimeError(f\'TECH_VALIDATION:{errors[:5]}\')\n    byid={x[\'item_id\']:x for x in gear[\'items\']}\n    for gid in GEARS:\n        count=collections.Counter()\n        for iid in selected_gear(gid):\n            if iid not in byid:raise RuntimeError(f\'MISSING_GEAR:{gid}:{iid}\')\n            it=byid[iid]\n            if it[\'min_stage\'] not in (\'LianQi_I\',\'LianQi_II\'):raise RuntimeError(f\'ILLEGAL_GEAR:{iid}\')\n            count[it[\'slot\']]+=1\n        if any(n>gear[\'slots\'][sl] for sl,n in count.items()):raise RuntimeError(f\'GEAR_SLOT_CONFLICT:{gid}\')\n    for mid in SPECIES:\n        profile=reg[\'profiles\'][mid];t0=f[\'T0\'][\'profiles\'][mid];ab=f[\'T1\'][\'species\'][mid][\'ability\']\n        assert profile[\'stats_status\']==\'READY\' and profile[\'native_stage\']==STAGE\n        # T0 numeric/mechanical freeze must be exact. The live registry may\n        # additionally mark a READY technique with administrative params_status.\n        # This is the ONLY metadata key allowed to differ from the frozen T0.\n        if profile[\'stats\'] != t0[\'stats\']:\n            raise RuntimeError(f\'T0_STATS_DRIFT:{mid}\')\n        reg_tech = copy.deepcopy(profile[\'technique\'])\n        if reg_tech.pop(\'params_status\', None) != \'READY\':\n            raise RuntimeError(f\'TECHNIQUE_STATUS_NOT_READY:{mid}\')\n        if reg_tech != t0[\'technique\']:\n            raise RuntimeError(f\'T0_TECHNIQUE_DRIFT:{mid}\')\n        assert profile[\'role\'] not in (\'BOSS\',\'ELITE\')\n        assert profile[\'adaptive\'][\'status\']==\'READY_FOR_T3_RECALIBRATION\'\n        assert ab[\'cooldown_rounds\']==3\n        if mid==\'sapo_ceniza\':\n            assert ab[\'kind\']==\'MITIGATE_NEXT\' and ab[\'damage_reduction_pct\']==60\n            assert ab[\'consumption\']==\'NEXT_CONNECTED_DIRECT_PACKET\' and ab[\'persists_through_miss\']\n            assert ab[\'affects_dot\'] is False\n        else:\n            assert ab[\'kind\']==\'DEFENSE_UP\' and ab[\'defense_bonus\']==4\n            assert ab[\'consumption\']==\'NEXT_PLAYER_OFFENSIVE_ACTION\'\n            assert ab[\'normal_penetration_defense_pipeline\'] is True\n    # Fail closed if local combat engine no longer delegates these survival behaviors.\n    fn=inspect.getsource(eng.resolve_player_direct)\n    consume=inspect.getsource(eng._consume_monster_survival_after_player_action)\n    dfn=inspect.getsource(eng._monster_dynamic_defense)\n    cooldown=inspect.getsource(eng.end_round)\n    assert \'MITIGATE_NEXT\' in fn and \'_consume_monster_survival_after_player_action\' in fn\n    assert \'DEFENSE_UP\' in consume and \'MITIGATE_NEXT\' in consume\n    assert \'DEFENSE_UP\' in dfn and \'monster_survival_cd\' in cooldown\n    compiled=0\n    for root in ROOTS:\n        for bid,paths,points in builds_for_root(root):\n            if points>2:raise RuntimeError(\'POINT_BUDGET\')\n            for tid in (OFF[root],DEF[root]):\n                comp=eng.compile_technique(tech[\'techniques\'][tid],paths.get(tid,()),root)\n                assert len(comp[\'families\'])<=1\n                assert comp[\'targeting\'] in (\'SELF\',\'UNITARGET\')\n                assert comp[\'role\'] in (\'DEFENSIVE\',\'OFFENSIVE\',\'CONTROL\')\n                compiled+=1\n    return {\'status\':\'PASS\',\'species\':list(SPECIES),\'freeze_status\':{k:v[\'status\'] for k,v in f.items()},\n            \'builds\':5*16,\'compiled\':compiled,\'A0\':0,\'point_limit\':2,\n            \'input_hashes_source\':\'PINNED_GIT_BLOBS\'}\n\n\nclass Recognition:\n    """Strict two-effective-action consecutive-category recognition."""\n    def __init__(self):\n        self.memory=collections.deque(maxlen=2)\n        self.trigger_count=0\n    def observe(self,category,result):\n        if category not in G[\'freeze\'][\'T2\'][\'candidate\'][\'observable_categories\']:\n            raise RuntimeError(f\'INVALID_OBSERVABLE:{category}\')\n        if result not in (\'EFECTIVA\',\'FALLIDA\'):raise RuntimeError(f\'INVALID_RESULT:{result}\')\n        self.memory.append((category,result))\n        if len(self.memory)==2 and all(x[1]==\'EFECTIVA\' for x in self.memory) and self.memory[0][0]==self.memory[1][0]:\n            self.trigger_count+=1\n            return category\n        return None\n\n\ndef arm_t1(state,mid):\n    cfg=G[\'freeze\'][\'T1\'][\'species\'][mid][\'ability\']\n    if cfg[\'kind\']==\'MITIGATE_NEXT\':\n        state.monster_survival={\'kind\':cfg[\'kind\'],\'damage_reduction_pct\':float(cfg[\'damage_reduction_pct\'])}\n    elif cfg[\'kind\']==\'DEFENSE_UP\':\n        state.monster_survival={\'kind\':cfg[\'kind\'],\'defense_bonus\':float(cfg[\'defense_bonus\'])}\n    else:raise RuntimeError(\'UNKNOWN_T1_KIND\')\n    state.monster_survival_cd=int(cfg[\'cooldown_rounds\'])\n    state.metrics.adaptation_procs+=1\n\n\ndef maybe_arm(state,mid,tier,recognized,direct_hp_damage):\n    if tier==\'T0\' or not state.monster.alive():return None\n    ratio=state.monster.hp/max(1.0,state.monster.hp_max)\n    packet=direct_hp_damage/max(1.0,state.monster.hp_max)\n    t1=G[\'freeze\'][\'T1\'][\'semantics\'][\'natural_trigger\']\n    c=G[\'freeze\'][\'T2\'][\'candidate\'][\'anticipation\']\n    natural=ratio<=t1[\'hp_ratio_lte\'] or packet>=t1[\'received_hit_ratio_gte\']\n    anticipatory=(tier==\'T2\' and recognized is not None and\n                   (ratio<=c[\'hp_ratio_lte\'] or packet>=c[\'received_hit_ratio_gte\']))\n    if state.monster_survival is not None or state.monster_survival_cd>0:return None\n    if natural:arm_t1(state,mid);return \'NATURAL\'\n    if anticipatory:arm_t1(state,mid);return \'ANTICIPATORY\'\n    return None\n\n\ndef make_state(root,paths,gid,mid,tier):\n    eng=G[\'eng\'];r=G[\'registry\'][\'profiles\'][mid]\n    player,meta=eng.build_player(STAGE,root,selected_gear(gid),G[\'gear\'])\n    # The source engine itself only allows building monster T0. Tier is held by\n    # this *lab adapter* as FightState metadata, not passed to build_monster.\n    monster=eng.build_monster(r,\'T0\')\n    compiled=eng.compile_build(root,eng.normalize_paths(paths),G[\'tech\'])\n    state=eng.FightState(player=player,monster=monster,player_root=root,stage=STAGE,tier=tier,\n                         player_basic_dice=eng.STAGES[STAGE][\'basic_attack\'],\n                         tech_scalar=eng.STAGES[STAGE][\'tech_scalar\'],compiled=compiled,\n                         monster_profile=r,signal_bridge=eng.LabSignalBridge(low_hp_ratio=.30,heavy_hit_ratio=.20),\n                         equipment_effects=meta[\'equipment_effects\'])\n    return state\n\n\ndef choose_action(state,policy):\n    if policy==\'UNITARGET_FIRST\':return OFF[state.player_root]\n    if policy==\'DEFENSE_OPEN\':return DEF[state.player_root] if state.round_no==1 else OFF[state.player_root]\n    raise RuntimeError(f\'INVALID_POLICY:{policy}\')\n\n\ndef take_player_action(state,rng,policy):\n    eng=G[\'eng\'];action=choose_action(state,policy);c=state.compiled[action]\n    hp_before=float(state.monster.hp)\n    performed=eng.execute_player_technique(state,rng,c)\n    if performed:\n        cat=\'PLAYER_DEFENSIVE_TECHNIQUE\' if c[\'role\']==\'DEFENSIVE\' else \'PLAYER_UNITARGET_TECHNIQUE\'\n        is_def=c[\'role\']==\'DEFENSIVE\'\n    else:\n        eng.execute_basic(state,rng)\n        cat=\'PLAYER_BASIC\';is_def=False\n    actual_damage=max(0.0,hp_before-float(state.monster.hp))\n    outcome=\'EFECTIVA\' if (is_def and performed) or actual_damage>0 else \'FALLIDA\'\n    return cat,outcome,actual_damage,is_def\n\n\ndef fight(job,seed):\n    root,build_id,paths,gid,mid,policy,tier=job\n    eng=G[\'eng\'];state=make_state(root,paths,gid,mid,tier)\n    rng=random.Random(seed)\n    reco=Recognition();natural=anticipatory=due=due_missed=def_actions=aoe_actions=0\n    for _ in range(MAX_ROUNDS):\n        if not state.player.alive() or not state.monster.alive():break\n        state.round_no+=1\n        eng.player_turn_start(state,rng)\n        if not state.player.alive():break\n        cat,outcome,damage,is_def=take_player_action(state,rng,policy)\n        def_actions+=int(is_def)\n        aoe_actions+=int(cat==\'PLAYER_AOE_TECHNIQUE\')\n        recognized=reco.observe(cat,outcome) if tier==\'T2\' else None\n        if not state.monster.alive():break\n        eng.monster_turn_start(state,rng)\n        if not state.monster.alive():break\n        reason=maybe_arm(state,mid,tier,recognized,damage)\n        natural+=int(reason==\'NATURAL\');anticipatory+=int(reason==\'ANTICIPATORY\')\n        # Activation must NOT replace a canonical technique due this round.\n        cfg=state.monster_profile[\'technique\'][\'params\'];cadence=int(cfg[\'cadence\'])\n        is_due=(state.round_no%cadence==0 and not state.monster.skip_next_action)\n        log_before=len(state.monster_recent_actions)\n        eng.execute_monster_turn(state,rng)\n        if is_due:\n            due+=1\n            if not state.monster_recent_actions or state.monster_recent_actions[-1]!=\'TECHNIQUE\':due_missed+=1\n        eng.end_round(state)\n    return {\'win\':int(state.monster.hp<=0 and state.player.hp>0),\n            \'timeout\':int(state.round_no>=MAX_ROUNDS and state.player.alive() and state.monster.alive()),\n            \'rounds\':state.round_no,\'hp_pct\':max(0,state.player.hp)/state.player.hp_max,\n            \'qi_final\':max(0,state.player.qi),\'qi_spent\':state.metrics.qi_spent,\n            \'defensive_actions\':def_actions,\'aoe_actions\':aoe_actions,\n            \'natural\':natural,\'anticipatory\':anticipatory,\'recognitions\':reco.trigger_count,\n            \'due\':due,\'due_missed\':due_missed}\n\n\ndef event_tests():\n    """Actual engine-level observations, not text-only assertions."""\n    eng=G[\'eng\'];results={}\n    reco=Recognition()\n    assert reco.observe(\'PLAYER_DEFENSIVE_TECHNIQUE\',\'EFECTIVA\') is None\n    assert reco.observe(\'PLAYER_UNITARGET_TECHNIQUE\',\'EFECTIVA\') is None\n    assert reco.observe(\'PLAYER_UNITARGET_TECHNIQUE\',\'FALLIDA\') is None\n    assert reco.observe(\'PLAYER_UNITARGET_TECHNIQUE\',\'EFECTIVA\') is None\n    assert reco.observe(\'PLAYER_UNITARGET_TECHNIQUE\',\'EFECTIVA\')==\'PLAYER_UNITARGET_TECHNIQUE\'\n    results[\'recognition_consecutive_2_effective\']=\'PASS\'\n    \n    for mid in SPECIES:\n        state=make_state(\'fuego\',{},\'MANDATORY_ENTRY\',mid,\'T2\')\n        state.monster.hp=.35*state.monster.hp_max;state.round_no=2\n        # Effective recognition at HP35% arms anticipatory T1, not natural T1.\n        reason=maybe_arm(state,mid,\'T2\',\'PLAYER_UNITARGET_TECHNIQUE\',0.0)\n        assert reason==\'ANTICIPATORY\' and state.monster_survival_cd==3\n        assert maybe_arm(state,mid,\'T2\',\'PLAYER_UNITARGET_TECHNIQUE\',0.0) is None\n        results[mid+\'_anticipation_35pct\']=\'PASS\'\n        other=make_state(\'fuego\',{},\'MANDATORY_ENTRY\',mid,\'T2\')\n        other.monster.hp=.25*other.monster.hp_max;other.round_no=3\n        reason=maybe_arm(other,mid,\'T2\',\'PLAYER_UNITARGET_TECHNIQUE\',0.25*other.monster.hp_max)\n        assert reason==\'NATURAL\'  # natural always wins priority\n        old=other.monster_survival_cd\n        assert old==3\n        before=len(other.monster_recent_actions)\n        eng.execute_monster_turn(other,random.Random(20261007))\n        assert len(other.monster_recent_actions)==before+1\n        assert other.monster_recent_actions[-1]==\'TECHNIQUE\'\n        assert other.monster_survival_cd==3\n        eng.end_round(other)\n        assert other.monster_survival_cd==2\n        results[mid+\'_natural_priority_cadence_due_and_cd\']=\'PASS\'\n\n        # DIRECT hit/miss probe on the real native engine.\n        base=make_state(\'fuego\',{},\'MANDATORY_ENTRY\',mid,\'T1\')\n        base.player_basic_dice=\'1d1+20\'  # fixture only; not a canonical monster/gear change\n        base.player.crit_chance=0\n        base.player.precision=1000\n        no=copy.deepcopy(base);on=copy.deepcopy(base)\n        arm_t1(on,mid)\n        rr0=eng.resolve_player_direct(no,random.Random(9),None,True)\n        rr1=eng.resolve_player_direct(on,random.Random(9),None,True)\n        assert rr0[\'hit\'] and rr1[\'hit\']\n        assert rr1[\'actual_hp_damage\']<rr0[\'actual_hp_damage\']\n        assert on.monster_survival is None\n        miss=make_state(\'fuego\',{},\'MANDATORY_ENTRY\',mid,\'T1\')\n        miss.player.precision=-1000\n        arm_t1(miss,mid)\n        rm=eng.resolve_player_direct(miss,random.Random(0),None,True)\n        assert not rm[\'hit\']\n        if mid==\'sapo_ceniza\': assert miss.monster_survival is not None\n        else: assert miss.monster_survival is None\n        results[mid+\'_connected_packet_and_miss_consumption\']=\'PASS\'\n\n        # A player defense can be effective, costs Qi, and does not itself\n        # consume monster survival on the offensive packet pipeline.\n        ds=make_state(\'fuego\',{},\'MANDATORY_ENTRY\',mid,\'T2\')\n        arm_t1(ds,mid); qi=ds.player.qi\n        ok=eng.execute_player_technique(ds,random.Random(1),ds.compiled[\'cuerpo_horno\'])\n        assert ok and ds.player.qi<qi and ds.monster_survival is not None\n        results[mid+\'_defense_activation_and_qi\']=\'PASS\'\n\n    # T0 control is immune to T1/T2 adapter arms.\n    fresh=make_state(\'fuego\',{},\'MANDATORY_ENTRY\',\'sapo_ceniza\',\'T0\')\n    fresh.monster.hp=.20*fresh.monster.hp_max\n    assert maybe_arm(fresh,\'sapo_ceniza\',\'T0\',\'PLAYER_BASIC\',40.0) is None\n    assert fresh.monster_survival is None\n    results[\'T0_does_not_inherit_adaptation\']=\'PASS\'\n    return results\n\n\nclass Progress:\n    """Single HBox/IntProgress per phase, mutate-only during updates."""\n    COLORS={\'RUN\':\'#3B82F6\',\'CP\':\'#EAB308\',\'DONE\':\'#22C55E\',\'ERR\':\'#EF4444\'}\n    def __init__(self,total,title):\n        self.n=0;self.total=total;self.title=title;self.start=time.monotonic();self.last=-1\n        self.enabled=False\n        try:\n            import ipywidgets as W\n            from IPython.display import display\n            self.W=W;self.enabled=True\n            self.tag=W.HTML(layout=W.Layout(width=\'52px\'))\n            self.bar=W.IntProgress(min=0,max=total,value=0,layout=W.Layout(width=\'300px\'))\n            self.detail=W.HTML(layout=W.Layout(width=\'245px\'))\n            self.box=W.HBox([W.HTML(\'<b>\'+title+\'</b>\',layout=W.Layout(width=\'155px\')),self.tag,self.bar,self.detail],\n                            layout=W.Layout(align_items=\'center\',width=\'850px\',min_width=\'850px\',gap=\'9px\'))\n            display(self.box)  # once only\n        except Exception:\n            print(title+\' progress (console fallback)\')\n        self.update(0,force=True)\n    def _paint(self,state):\n        if self.enabled:\n            c=self.COLORS[state];self.tag.value=\'<span style="font:700 11px monospace;color:\'+c+\'">\'+state+\'</span>\'\n            self.bar.style.bar_color=c\n            self.bar.value=self.n\n            seconds=max(1e-6,time.monotonic()-self.start)\n            rate=self.n/seconds\n            rest=(self.total-self.n)/rate if rate else 0\n            self.detail.value=\'<span style="white-space:nowrap;font:12px monospace">\'+f\'{100*self.n/self.total:5.1f}%  {self.n:,}/{self.total:,}  ETA {int(rest):d}s\'+\'</span>\'\n        elif state in (\'DONE\',\'ERR\') or self.n==0 or self.n==self.total:\n            print(state, f\'{self.n:,}/{self.total:,}\',flush=True)\n    def update(self,delta,force=False):\n        self.n=min(self.total,self.n+delta)\n        now=time.monotonic()\n        if force or now-self.last>=.25 or self.n==self.total:self.last=now;self._paint(\'RUN\')\n    def cp(self):self._paint(\'CP\')\n    def resume(self):self._paint(\'RUN\')\n    def done(self):self.n=self.total;self._paint(\'DONE\')\n    def err(self):self._paint(\'ERR\')\n\n\ndef worker_init(repo):load_sources(repo);validate_contracts()\n\n\ndef do_job(item):\n    (root,bid,paths,gid,mid,pol,tier),r=item\n    win=hp=qi=sp=turns=defs=aoe=nat=ant=rec=due=miss=timeout=0\n    # Seed excludes tier for paired T0/T1/T2 differences on same starting RNG.\n    for i in range(r):\n        seed=stable_seed(root,bid,gid,mid,pol,i)\n        x=fight((root,bid,paths,gid,mid,pol,tier),seed)\n        win+=x[\'win\'];hp+=x[\'hp_pct\'];qi+=x[\'qi_final\'];sp+=x[\'qi_spent\'];turns+=x[\'rounds\']\n        defs+=x[\'defensive_actions\'];aoe+=x[\'aoe_actions\']\n        nat+=x[\'natural\'];ant+=x[\'anticipatory\'];rec+=x[\'recognitions\']\n        due+=x[\'due\'];miss+=x[\'due_missed\'];timeout+=x[\'timeout\']\n    return {\'root\':root,\'build_id\':bid,\'gear\':gid,\'monster\':mid,\'policy\':pol,\'tier\':tier,\'R\':r,\n            \'win_rate\':win/r,\'hp_pct\':hp/r,\'qi_final\':qi/r,\'qi_spent\':sp/r,\'rounds\':turns/r,\n            \'defense_uses\':defs,\'aoe_uses\':aoe,\'natural_activations\':nat,\'anticipatory_activations\':ant,\n            \'recognitions\':rec,\'canonical_due\':due,\'canonical_due_missed\':miss,\'timeouts\':timeout}\n\n\ndef enumerate_jobs():\n    jobs=[]\n    for root in ROOTS:\n        for bid,paths,points in builds_for_root(root):\n            for gid in GEARS:\n                for mid in SPECIES:\n                    for pol in POLICIES:\n                        for tier in ARMS:jobs.append(((root,bid,paths,gid,mid,pol,tier),R))\n    assert len(jobs)==5*16*4*2*2*3\n    return jobs\n\n\ndef run(args=None):\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--repo\',required=True)\n    ap.add_argument(\'--out\',required=True)\n    ap.add_argument(\'--runner\',required=True)\n    ap.add_argument(\'--workers\',type=int,default=WORKERS_DEFAULT)\n    a=ap.parse_args(args)\n    out=Path(a.out).resolve();out.mkdir(parents=True,exist_ok=True)\n    sources=verify_sources(a.repo)\n    save_json(out/\'SOURCE_LOCK.json\',sources)\n    load_sources(a.repo)\n    contract=validate_contracts();save_json(out/\'PREFLIGHT.json\',contract)\n    ev=event_tests();save_json(out/\'EVENT_PARITY_TESTS.json\',ev)\n    jobs=enumerate_jobs();chunk_size=64\n    chunks=[jobs[i:i+chunk_size] for i in range(0,len(jobs),chunk_size)]\n    cp=out/\'checkpoints\';cp.mkdir(exist_ok=True)\n    pb=Progress(len(jobs)*R,\'Bridge T0/T1/T2\')\n    rows=[]; cached=0\n    try:\n        for i,chunk in enumerate(chunks):\n            fname=cp/f\'chunk_{i:04d}.jsonl.gz\'\n            if fname.exists():\n                pb.cp()\n                with gzip.open(fname,\'rt\',encoding=\'utf-8\') as f: r=[json.loads(line) for line in f if line.strip()]\n                if len(r)!=len(chunk):raise RuntimeError(f\'INVALID_CHECKPOINT_ROWS:{fname}\')\n                cached+=1;rows.extend(r)\n                pb.resume();pb.update(len(chunk)*R)\n                continue\n            if a.workers==1:r=[do_job(x) for x in chunk]\n            else:\n                with ProcessPoolExecutor(max_workers=a.workers,initializer=worker_init,initargs=(str(a.repo),)) as pool:\n                    r=list(pool.map(do_job,chunk))\n            # Only write complete chunks; failed chunks don\'t contaminate resume.\n            pb.cp();tmp=fname.with_suffix(fname.suffix+\'.tmp\')\n            with gzip.open(tmp,\'wt\',encoding=\'utf-8\') as f:\n                for row in r:f.write(json.dumps(row,sort_keys=True)+\'\\n\')\n            tmp.replace(fname);rows.extend(r);pb.resume();pb.update(len(chunk)*R)\n        pb.done()\n    except Exception:pb.err();raise\n    expected=len(jobs)\n    if len(rows)!=expected:raise RuntimeError(f\'ROW_COUNT:{len(rows)}:{expected}\')\n    import pandas as pd\n    df=pd.DataFrame(rows)\n    if df.duplicated([\'root\',\'build_id\',\'gear\',\'monster\',\'policy\',\'tier\']).any():raise RuntimeError(\'DUPLICATE_CONTEXT\')\n    num=df.select_dtypes(include=\'number\').to_numpy()\n    if not __import__(\'numpy\').isfinite(num).all():raise RuntimeError(\'NONFINITE_NUMBERS\')\n    if df[\'canonical_due_missed\'].sum()>0:raise RuntimeError(\'CANONICAL_DUE_SUPPRESSED\')\n    if df[\'aoe_uses\'].sum()>0:raise RuntimeError(\'ILLEGAL_AOE_USED\')\n    if df[\'timeouts\'].sum()>0:raise RuntimeError(\'TIMEOUTS_IN_SMOKE\')\n    if df.query(\'tier == "T0"\')[\'natural_activations\'].sum()>0:raise RuntimeError(\'T0_ACTIVATIONS\')\n    if df.query(\'tier == "T1"\')[\'anticipatory_activations\'].sum()>0:raise RuntimeError(\'T1_ANTICIPATIONS\')\n    if df.query(\'tier == "T2"\')[\'anticipatory_activations\'].sum()<=0:raise RuntimeError(\'T2_NEVER_ANTICIPATES\')\n    df.to_csv(out/\'BRIDGE_SMOKE_CONTEXTS.csv.gz\',compression=\'gzip\',index=False)\n    summary=df.groupby([\'monster\',\'tier\',\'gear\',\'policy\'],as_index=False).agg(\n        win_rate=(\'win_rate\',\'mean\'),hp_pct=(\'hp_pct\',\'mean\'),qi_spent=(\'qi_spent\',\'mean\'),\n        natural=(\'natural_activations\',\'sum\'),anticipatory=(\'anticipatory_activations\',\'sum\'),\n        due=(\'canonical_due\',\'sum\'),due_missed=(\'canonical_due_missed\',\'sum\'),contexts=(\'R\',\'size\'))\n    summary.to_csv(out/\'BRIDGE_SMOKE_SUMMARY.csv\',index=False)\n    verdict={\'status\':\'PASS_BRIDGE_ONLY\',\'issues\':[],\'represented_fights\':len(jobs)*R,\n        \'contexts\':len(jobs),\'R\':R,\'workers\':a.workers,\'cached_chunks\':cached,\'checkpoints\':len(chunks),\n        \'event_tests\':len(ev),\'total_t1_natural\':int(df.query(\'tier == "T1"\')[\'natural_activations\'].sum()),\n        \'total_t2_anticipations\':int(df.query(\'tier == "T2"\')[\'anticipatory_activations\'].sum()),\n        \'due_missed\':int(df[\'canonical_due_missed\'].sum()),\'aoe_uses\':int(df[\'aoe_uses\'].sum()),\n        \'note\':\'Only bridge parity/smoke; not LII balance freeze; no monster AI or stat changes.\',\n        \'next_gate\':\'LII_DEFENSIVE_T1_T2_CAUSAL_GATE_V01\'}\n    save_json(out/\'DECISION_GATES.json\',verdict)\n    cp_hash={p.name:sha256(p) for p in sorted(cp.glob(\'chunk_*.jsonl.gz\'))}\n    save_json(out/\'CHECKPOINT_HASHES.json\',cp_hash)\n    shutil.copy2(a.runner,out/\'LAB_RUNNER.py\')\n    files=[\'SOURCE_LOCK.json\',\'PREFLIGHT.json\',\'EVENT_PARITY_TESTS.json\',\'BRIDGE_SMOKE_CONTEXTS.csv.gz\',\n           \'BRIDGE_SMOKE_SUMMARY.csv\',\'DECISION_GATES.json\',\'CHECKPOINT_HASHES.json\',\'LAB_RUNNER.py\']\n    manifest={f:{\'sha256\':sha256(out/f),\'size\':(out/f).stat().st_size} for f in files}\n    save_json(out/\'PACKAGE_MANIFEST.json\',{\'files\':manifest,\'total\':len(files)})\n    z=out/(LAB_ID+\'_REVIEW.zip\')\n    with zipfile.ZipFile(z,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=8) as w:\n        for f in [*files,\'PACKAGE_MANIFEST.json\']:w.write(out/f,f)\n    print(json.dumps({\'status\':verdict[\'status\'],\'review\':str(z),\'zip_sha256\':sha256(z),\n                      \'represented_fights\':verdict[\'represented_fights\'],\'cached_chunks\':cached},indent=2),flush=True)\n    return z\n\n\n\n\n# ==== LII DEFENSIVE T1/T2 CAUSAL SCREEN ==== \nPOLICIES_CAUSAL=(\'OFFENSE_ONLY\',\'DEFENSE_OPEN\',\'DEFENSE_DUE\',\'DEFENSE_GUARD\')\nR_CAUSAL=32\nCAUSAL_CHUNK_SIZE=32\n\n\ndef causal_policy_decision(state,policy,def_casts):\n    """Only observed state and published cadence, never future RNG."""\n    if policy==\'OFFENSE_ONLY\':return OFF[state.player_root]\n    if policy==\'DEFENSE_OPEN\':\n        return DEF[state.player_root] if state.round_no==1 else OFF[state.player_root]\n    if policy not in (\'DEFENSE_DUE\',\'DEFENSE_GUARD\'):\n        raise RuntimeError(f\'UNKNOWN_POLICY:{policy}\')\n    if state.player_defense is not None or def_casts >= (1 if policy==\'DEFENSE_DUE\' else 2):\n        return OFF[state.player_root]\n    dc=state.compiled[DEF[state.player_root]][\'qi_cost\']\n    oc=state.compiled[OFF[state.player_root]][\'qi_cost\']\n    if state.player.qi < dc+oc:return OFF[state.player_root]\n    if state.monster.hp/max(1.,state.monster.hp_max)<.25:\n        return OFF[state.player_root]\n    cadence=int(state.monster_profile[\'technique\'][\'params\'][\'cadence\'])\n    due=(state.round_no%cadence==0)\n    if policy==\'DEFENSE_DUE\':\n        return DEF[state.player_root] if due else OFF[state.player_root]\n    hp_low=(state.player.hp/max(1.,state.player.hp_max)<=.70)\n    return DEF[state.player_root] if (due or hp_low) else OFF[state.player_root]\n\n\ndef causal_fight(root,paths,gid,mid,tier,policy,seed):\n    eng=G[\'eng\'];state=make_state(root,paths,gid,mid,tier)\n    rng=random.Random(seed)\n    reco=Recognition();natural=anticipatory=due=due_missed=def_acts=aoe_acts=0\n    fallback_basics=0\n    for _ in range(MAX_ROUNDS):\n        if not (state.player.alive() and state.monster.alive()):break\n        state.round_no+=1\n        eng.player_turn_start(state,rng)\n        if not state.player.alive():break\n        tid=causal_policy_decision(state,policy,def_acts)\n        c=state.compiled[tid]\n        before=float(state.monster.hp)\n        performed=eng.execute_player_technique(state,rng,c)\n        if not performed:\n            eng.execute_basic(state,rng)\n            cat=\'PLAYER_BASIC\';is_def=False;fallback_basics+=1\n        else:\n            cat=\'PLAYER_DEFENSIVE_TECHNIQUE\' if c[\'role\']==\'DEFENSIVE\' else \'PLAYER_UNITARGET_TECHNIQUE\'\n            is_def=(c[\'role\']==\'DEFENSIVE\')\n        def_acts+=int(is_def)\n        aoe_acts+=int(cat==\'PLAYER_AOE_TECHNIQUE\')\n        direct_delta=max(0.,before-float(state.monster.hp))\n        result=\'EFECTIVA\' if (is_def or direct_delta>0.) else \'FALLIDA\'\n        recognized=reco.observe(cat,result) if tier==\'T2\' else None\n        if not state.monster.alive():break\n        eng.monster_turn_start(state,rng)\n        if not state.monster.alive():break\n        reason=maybe_arm(state,mid,tier,recognized,direct_delta)\n        natural+=int(reason==\'NATURAL\');anticipatory+=int(reason==\'ANTICIPATORY\')\n        params=state.monster_profile[\'technique\'][\'params\']\n        this_due=(state.round_no%int(params[\'cadence\'])==0 and not state.monster.skip_next_action)\n        eng.execute_monster_turn(state,rng)\n        if this_due:\n            due+=1\n            if not state.monster_recent_actions or state.monster_recent_actions[-1]!=\'TECHNIQUE\':due_missed+=1\n        eng.end_round(state)\n    m=state.metrics\n    return {\n        \'win\':int(state.monster.hp<=0 and state.player.hp>0),\n        \'hp_pct\':max(0.,state.player.hp)/state.player.hp_max,\n        \'qi_final\':max(0.,state.player.qi),\n        \'qi_spent\':float(m.qi_spent),\n        \'rounds\':int(state.round_no),\n        \'defense_uses\':def_acts,\n        \'defense_procs\':int(m.defense_procs),\n        \'player_def_prevented\':float(m.player_def_prevented),\n        \'player_absorbed\':float(m.player_absorbed),\n        \'player_evades\':int(m.player_evades),\n        \'qi_drained\':float(m.qi_drained),\n        \'monster_damage_direct\':float(m.monster_damage_direct),\n        \'monster_damage_dot\':float(m.monster_damage_dot),\n        \'player_damage_direct\':float(m.player_damage_direct),\n        \'player_damage_dot\':float(m.player_damage_dot),\n        \'control_successes\':int(m.control_successes),\n        \'natural\':natural,\'anticipatory\':anticipatory,\'recognitions\':reco.trigger_count,\n        \'canonical_due\':due,\'canonical_due_missed\':due_missed,\n        \'fallback_basics\':fallback_basics,\n        \'aoe_uses\':aoe_acts,\n        \'timeout\':int(state.round_no>=MAX_ROUNDS and state.player.alive() and state.monster.alive()),\n    }\n\n\ndef causal_jobs():\n    jobs=[]\n    for root in ROOTS:\n        for build_id,paths,points in builds_for_root(root):\n            assert points<=2\n            for gid in GEARS:\n                for mid in SPECIES:\n                    for tier in (\'T1\',\'T2\'):\n                        jobs.append((root,build_id,paths,gid,mid,tier))\n    assert len(jobs)==1280\n    return jobs\n\n\ndef causal_do_job(job):\n    root,build_id,paths,gid,mid,tier=job\n    rows=[]\n    # EXACT common random seeds across policies and comparable T1 build choices.\n    for i in range(R_CAUSAL):\n        seed=stable_seed(\'CAUSAL\',root,gid,mid,tier,i)\n        for policy in POLICIES_CAUSAL:\n            v=causal_fight(root,paths,gid,mid,tier,policy,seed)\n            rows.append({\'root\':root,\'build_id\':build_id,\'gear\':gid,\'monster\':mid,\n                         \'tier\':tier,\'policy\':policy,\'replicate\':i,**v})\n    return rows\n\n\ndef causal_paired_summary(df,compare,base_keys,labels=(\'baseline\',\'treatment\')):\n    import pandas as pd\n    import numpy as np\n    keys=[*base_keys,\'replicate\']\n    aa,bb=compare\n    if len(aa)!=len(bb):raise RuntimeError(\'UNBALANCED_COMPARISON\')\n    a=aa.set_index(keys).sort_index();b=bb.set_index(keys).sort_index()\n    if not a.index.equals(b.index):raise RuntimeError(\'PAIR_KEYS_MISMATCH\')\n    cols=(\'win\',\'hp_pct\',\'qi_final\',\'qi_spent\',\'rounds\',\'defense_uses\',\'defense_procs\',\'player_absorbed\',\'player_evades\',\'qi_drained\',\'natural\',\'anticipatory\')\n    delta=b.loc[:,cols]-a.loc[:,cols]\n    scalar=[\'win\',\'hp_pct\']\n    groups=delta.groupby(level=base_keys,sort=True)\n    out=[]\n    for id_,g in groups:\n        if not isinstance(id_,tuple):id_=(id_,)\n        vals=dict(zip(base_keys,id_))\n        vals[\'R\']=len(g)\n        for x in cols:\n            mu=float(g[x].mean())\n            scale=100 if x in scalar else 1\n            vals[\'delta_\'+x+(\'_pp\' if x in scalar else \'\')]=scale*mu\n            if x in (\'win\',\'hp_pct\'):\n                se=float(g[x].std(ddof=1)/math.sqrt(len(g))) if len(g)>1 else 0.0\n                vals[\'delta_\'+x+\'_ci95_low_pp\']=100*(mu-1.96*se)\n                vals[\'delta_\'+x+\'_ci95_high_pp\']=100*(mu+1.96*se)\n        vals[\'off_win_rate\']=float(a.loc[g.index,\'win\'].mean())\n        vals[\'on_win_rate\']=float(b.loc[g.index,\'win\'].mean())\n        vals[\'discordant_gain\']=int(((a.loc[g.index,\'win\']==0)&(b.loc[g.index,\'win\']==1)).sum())\n        vals[\'discordant_loss\']=int(((a.loc[g.index,\'win\']==1)&(b.loc[g.index,\'win\']==0)).sum())\n        out.append(vals)\n    return pd.DataFrame(out)\n\n\ndef summarize_causal(df):\n    import pandas as pd\n    idx=[\'root\',\'build_id\',\'gear\',\'monster\',\'tier\',\'replicate\']\n    baseline=df.loc[df.policy==\'OFFENSE_ONLY\'].copy()\n    policy_parts=[]\n    for other in POLICIES_CAUSAL[1:]:\n        selected=df.loc[df.policy==other].copy()\n        p=causal_paired_summary(df,(baseline,selected),idx[:-1])\n        p.insert(0,\'policy\',other)\n        policy_parts.append(p)\n    policy_pairs=pd.concat(policy_parts,ignore_index=True)\n\n    # Incremental Tramo I effects holding strategy/element/gear/monster/tier fixed.\n    # BASE -> defensive and offensive T1 nodes; OFF_T1_i -> OFF_i_DEF_j.\n    build_refs={}\n    for root in ROOTS:\n        build_refs[(root,\'BASE_0\')]=\'BASE_0\'\n        for i in range(3):\n            build_refs[(root,f\'OFF_T1_{i}\')]=\'BASE_0\'\n            build_refs[(root,f\'DEF_T1_{i}\')]=\'BASE_0\'\n            for j in range(3):build_refs[(root,f\'OFF_{i}_DEF_{j}\')]=f\'OFF_T1_{i}\'\n    pairs=[]\n    common_keys=[\'root\',\'gear\',\'monster\',\'tier\',\'policy\',\'replicate\']\n    for root in ROOTS:\n        dfr=df.loc[df.root==root]\n        for build_id in [b for b,_,_ in builds_for_root(root) if b!=\'BASE_0\']:\n            baseline_id=build_refs[(root,build_id)]\n            source=dfr.loc[dfr.build_id==baseline_id]\n            selected=dfr.loc[dfr.build_id==build_id]\n            p=causal_paired_summary(df,(source,selected),common_keys[:-1])\n            p.insert(0,\'root_label\',root)\n            p.insert(1,\'build_base\',baseline_id)\n            p.insert(2,\'build_variant\',build_id)\n            p.insert(3,\'branch_kind\',\'DEF_T1\' if build_id.startswith(\'DEF_T1\') or \'_DEF_\' in build_id else \'OFF_T1\')\n            pairs.append(p)\n    spec_pairs=pd.concat(pairs,ignore_index=True)\n\n    # Aggregate win/HP/turns/usage by policy without implying equal probability of builds/gear.\n    groups=[\'root\',\'monster\',\'tier\',\'gear\',\'policy\']\n    summary=df.groupby(groups,dropna=False,as_index=False).agg(\n        fights=(\'win\',\'size\'),win_rate=(\'win\',\'mean\'),hp_pct=(\'hp_pct\',\'mean\'),\n        qi_spent=(\'qi_spent\',\'mean\'),qi_final=(\'qi_final\',\'mean\'),rounds=(\'rounds\',\'mean\'),\n        defense_uses=(\'defense_uses\',\'mean\'),defense_procs=(\'defense_procs\',\'mean\'),\n        player_absorbed=(\'player_absorbed\',\'mean\'),player_evades=(\'player_evades\',\'mean\'),\n        player_def_prevented=(\'player_def_prevented\',\'mean\'),\n        natural=(\'natural\',\'mean\'),anticipatory=(\'anticipatory\',\'mean\'),\n    )\n    warning_mask=(policy_pairs.delta_win_pp<=-6.25) | ((policy_pairs.delta_win_pp<=0)&(policy_pairs.delta_hp_pct_pp<=-5.0))\n    warnings=policy_pairs.loc[warning_mask].sort_values(\'delta_win_pp\')\n    return policy_pairs,spec_pairs,summary,warnings\n\n\ndef preflight_causal():\n    pf=validate_contracts();ev=event_tests()\n    assert len(ev)==10 and all(v==\'PASS\' for v in ev.values())\n    assert len(causal_jobs())==1280\n    assert len(POLICIES_CAUSAL)==4 and R_CAUSAL==32\n    # Both defensive policies should preserve Qi for one offensive technique.\n    st=make_state(\'fuego\',{},\'MANDATORY_ENTRY\',\'sapo_ceniza\',\'T2\')\n    st.round_no=1\n    assert causal_policy_decision(st,\'OFFENSE_ONLY\',0)==OFF[\'fuego\']\n    assert causal_policy_decision(st,\'DEFENSE_OPEN\',0)==DEF[\'fuego\']\n    # No AOE / Tramo II gate; compiler covered by validate_contracts.\n    pf.update({\'status\':\'PASS_CAUSAL_SCREEN_PREFLIGHT\',\'event_tests\':len(ev),\n               \'policy_count\':4,\'base_contexts\':1280,\'R\':R_CAUSAL,\n               \'represented_fights\':1280*4*R_CAUSAL,\n               \'note\':\'R32 SCREEN ONLY; NO LII balance freeze; only 1-root builds.\'})\n    return pf,ev\n\n\ndef causal_main(args=None):\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--repo\',required=True)\n    ap.add_argument(\'--out\',required=True)\n    ap.add_argument(\'--runner\',required=True)\n    ap.add_argument(\'--workers\',type=int,default=2)\n    a=ap.parse_args(args)\n    out=Path(a.out).resolve();out.mkdir(parents=True,exist_ok=True)\n    sl=verify_sources(a.repo);save_json(out/\'SOURCE_LOCK.json\',sl)\n    load_sources(a.repo)\n    preflight,event_tests_data=preflight_causal()\n    save_json(out/\'PREFLIGHT.json\',preflight)\n    save_json(out/\'EVENT_PARITY_TESTS.json\',event_tests_data)\n    runner_hash=sha256(a.runner)\n    signature={\n        \'source_head\':sl[\'commit\'],\'source_blobs\':{k:v[\'blob_sha\'] for k,v in sl[\'files\'].items()},\n        \'runner_sha256\':runner_hash,\'policies\':list(POLICIES_CAUSAL),\n        \'R\':R_CAUSAL,\'gear\':list(GEARS),\'tiers\':[\'T1\',\'T2\'],\'species\':list(SPECIES),\n        \'builds\':\'80_MOONROOT_STAGE_II_MONOROOT\',\n    }\n    sig_file=out/\'CHECKPOINT_RUN_SIGNATURE.json\'\n    if sig_file.exists():\n        existing=json.loads(sig_file.read_text())\n        if existing!=signature:raise RuntimeError(\'CHECKPOINT_SIGNATURE_MISMATCH:USE_NEW_OUT_DIR\')\n    else:save_json(sig_file,signature)\n    jobs=causal_jobs()\n    chunks=[jobs[i:i+CAUSAL_CHUNK_SIZE] for i in range(0,len(jobs),CAUSAL_CHUNK_SIZE)]\n    cp=out/\'checkpoints\';cp.mkdir(exist_ok=True)\n    pb=Progress(len(jobs)*len(POLICIES_CAUSAL)*R_CAUSAL,\'LII Defense T1/T2 causal screen\')\n    rows=[];cached=0\n    try:\n        # Workers stay alive across chunks. No per-chunk repeated process startup.\n        with (ProcessPoolExecutor(max_workers=a.workers,initializer=worker_init,initargs=(str(a.repo),))\n              if a.workers>1 else __import__(\'contextlib\').nullcontext(None)) as pool:\n            for chunk_idx,chunk in enumerate(chunks):\n                file=cp/f\'chunk_{chunk_idx:04d}.jsonl.gz\'\n                if file.exists():\n                    pb.cp()\n                    with gzip.open(file,\'rt\',encoding=\'utf-8\') as f:rr=[json.loads(x) for x in f if x.strip()]\n                    if len(rr)!=len(chunk)*R_CAUSAL*len(POLICIES_CAUSAL):raise RuntimeError(f\'BAD_CACHE_COUNT:{file.name}\')\n                    need={(r,b,g,m,t) for r,b,_,g,m,t in chunk}\n                    have={(v[\'root\'],v[\'build_id\'],v[\'gear\'],v[\'monster\'],v[\'tier\']) for v in rr}\n                    if have!=need:raise RuntimeError(f\'BAD_CACHE_CONTEXTS:{file.name}\')\n                    cached+=1;rows.extend(rr);pb.resume();pb.update(len(rr));continue\n                if pool is None:\n                    items=[causal_do_job(x) for x in chunk]\n                else:\n                    items=list(pool.map(causal_do_job,chunk,chunksize=2))\n                rr=[v for sub in items for v in sub]\n                assert len(rr)==len(chunk)*R_CAUSAL*len(POLICIES_CAUSAL)\n                pb.cp();temp=file.with_suffix(file.suffix+\'.tmp\')\n                with gzip.open(temp,\'wt\',encoding=\'utf-8\') as f:\n                    for row in rr:f.write(json.dumps(row,sort_keys=True)+\'\\n\')\n                temp.replace(file)\n                rows.extend(rr);pb.resume();pb.update(len(rr))\n        pb.done()\n    except Exception:pb.err();raise\n    import pandas as pd,numpy as np\n    if len(rows)!=len(jobs)*len(POLICIES_CAUSAL)*R_CAUSAL:\n        raise RuntimeError(\'RAW_ROWS_MISMATCH\')\n    df=pd.DataFrame(rows)\n    keys=[\'root\',\'build_id\',\'gear\',\'monster\',\'tier\',\'policy\',\'replicate\']\n    if df.duplicated(keys).any():raise RuntimeError(\'DUPLICATE_SEED_KEY\')\n    num=df.select_dtypes(include=\'number\').to_numpy()\n    if not np.isfinite(num).all():raise RuntimeError(\'NONFINITE\')\n    if int(df[\'canonical_due_missed\'].sum())>0:raise RuntimeError(\'DUE_CANONICAL_MISSED\')\n    if int(df[\'aoe_uses\'].sum())>0:raise RuntimeError(\'AOE_LII_NOT_ALLOWED\')\n    if int(df[\'timeout\'].sum())>0:raise RuntimeError(\'TIMEOUT\')\n    df.to_csv(out/\'RAW_SEED_RESULTS.csv.gz\',index=False,compression=\'gzip\')\n    policy_pairs,spec_pairs,summary,warnings=summarize_causal(df)\n    policy_pairs.to_csv(out/\'POLICY_PAIR_CONTEXTS.csv.gz\',index=False,compression=\'gzip\')\n    spec_pairs.to_csv(out/\'TRAMO_I_PAIRED_COMPARISONS.csv.gz\',index=False,compression=\'gzip\')\n    summary.to_csv(out/\'POLICY_SUMMARY.csv\',index=False)\n    warnings.to_csv(out/\'FOCUS_WARNINGS.csv\',index=False)\n    diag={\n        \'status\':\'PASS_SCREEN_ONLY\',\'hard_issues\':[],\n        \'seed_level_rows\':int(len(df)),\n        \'represented_fights\':int(len(df)),\n        \'unique_scenarios\':len(jobs),\n        \'policies\':list(POLICIES_CAUSAL),\n        \'policy_pair_contexts\':int(len(policy_pairs)),\n        \'specialization_pair_contexts\':int(len(spec_pairs)),\n        \'focus_warning_contexts\':int(len(warnings)),\n        \'R\':R_CAUSAL,\'cached_chunks\':cached,\'checkpoints\':len(chunks),\n        \'canonical_due_missed\':int(df.canonical_due_missed.sum()),\n        \'aoe_uses\':int(df.aoe_uses.sum()),\n        \'timeouts\':int(df.timeout.sum()),\n        \'note\':\'Paired common-random-number policy interventions; different action sequences may desynchronize RNG draws. No claim of action-locked effects.\',\n        \'next\':\'AUDIT_R32_THEN_TARGETED_R256_NO_MASS_BROAD_RERUN\',\n    }\n    save_json(out/\'DECISION_GATES.json\',diag)\n    save_json(out/\'CHECKPOINT_HASHES.json\',{p.name:sha256(p) for p in sorted(cp.glob(\'chunk_*.jsonl.gz\'))})\n    shutil.copy2(a.runner,out/\'LAB_RUNNER.py\')\n    files=[\'SOURCE_LOCK.json\',\'PREFLIGHT.json\',\'EVENT_PARITY_TESTS.json\',\'CHECKPOINT_RUN_SIGNATURE.json\',\n           \'RAW_SEED_RESULTS.csv.gz\',\'POLICY_PAIR_CONTEXTS.csv.gz\',\'TRAMO_I_PAIRED_COMPARISONS.csv.gz\',\n           \'POLICY_SUMMARY.csv\',\'FOCUS_WARNINGS.csv\',\'DECISION_GATES.json\',\n           \'CHECKPOINT_HASHES.json\',\'LAB_RUNNER.py\']\n    man={\'files\':{f:{\'sha256\':sha256(out/f),\'size\':(out/f).stat().st_size} for f in files},\n         \'total\':len(files)}\n    save_json(out/\'PACKAGE_MANIFEST.json\',man)\n    z=out/(LAB_ID+\'_REVIEW.zip\')\n    with zipfile.ZipFile(z,\'w\',zipfile.ZIP_DEFLATED,compresslevel=8) as w:\n        for f in [*files,\'PACKAGE_MANIFEST.json\']:w.write(out/f,f)\n    print(json.dumps({\'status\':diag[\'status\'],\'review\':str(z),\n                      \'sha256\':sha256(z),\'fights\':len(df),\n                      \'warnings_for_followup\':len(warnings)},ensure_ascii=False,indent=2),flush=True)\n    return z\n\nif __name__==\'__main__\':causal_main()\n', 'LII_TRAMO_I_QI_BREAKPOINT_AND_POLICY_DIAG_V03_RUNNER.py': '"""La Grulla Blanca — LII Tramo I, Qi breakpoint and policy diagnostics V03.\nOnly laboratory; derived from the frozen LII T0/T1/T2 bridge. Never edits source files.\n"""\nfrom __future__ import annotations\nimport argparse, copy, gzip, hashlib, json, math, os, random, shutil, statistics, zipfile\nfrom pathlib import Path\nfrom concurrent.futures import ProcessPoolExecutor\nimport pandas as pd\nimport numpy as np\nimport LAB_RUNNER as b\n\nLAB_ID=\'LII_TRAMO_I_QI_BREAKPOINT_AND_POLICY_DIAG_V03\'\nSEED_TAG=\'FRESH_V03_CONFIRMATORY_20261007_NOT_V01_OR_V02\'\n# Type: PHASE, root, gear, monster, tier, build_id, policy, variant, R\nGEARS=(\'MANDATORY_ENTRY\',\'CARRY_OVER_LI_HIGH\',\'EXPECTED_STAGE\')\nTIERS=(\'T1\',\'T2\')\nPOLICIES_TIMING=(\'OFFENSE_ONLY\',\'DEFENSE_OPEN\',\'DEFENSE_DUE\',\'DEFENSE_GUARD\',\'THREAT_AWARE\')\nPOLICIES_WATER=(\'DEFENSE_OPEN\',\'DEFENSE_DUE\',\'DEFENSE_GUARD\')\nR_TIMING=128\nR_OFFENSE=64\nR_WATER=128\nROOTS=b.ROOTS\nSPECS=b.SPECIES\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(1<<20),b\'\'):h.update(chunk)\n    return h.hexdigest()\n\n\ndef initial_locks(repo,runner,base_runner):\n    lock=b.verify_sources(repo)\n    b.load_sources(repo)\n    contract=b.validate_contracts()\n    events=b.event_tests()\n    if len(events)!=10 or not all(x==\'PASS\' for x in events.values()):\n        raise RuntimeError(\'FROZEN_T1_T2_EVENT_PARITY_FAILURE\')\n    return lock,contract,events\n\n\ndef choice_paths(root,build_id):\n    d={name:(paths,pts) for name,paths,pts in b.builds_for_root(root)}\n    if build_id not in d:raise RuntimeError(f\'UNKNOWN_BUILD:{root}:{build_id}\')\n    p,points=d[build_id]\n    if points>2:raise RuntimeError(\'LII_POINT_BUDGET_EXCEEDED\')\n    return p\n\n\ndef build_jobs():\n    phases={\'TIMING\':[],\'OFFENSE\':[],\'WATER_QI\':[]}\n    # 2 roots × 3 gears × 2 monsters × 2 tiers × 3 builds × 5 policies × R128\n    for root in (\'fuego\',\'tierra\'):\n        ids=(\'BASE_0\',\'OFF_T1_0\',\'DEF_T1_1\') if root==\'fuego\' else (\'BASE_0\',\'OFF_T1_0\',\'DEF_T1_0\')\n        for gid in GEARS:\n            for mid in SPECS:\n                for tier in TIERS:\n                    for bid in ids:\n                        for policy in POLICIES_TIMING:\n                            phases[\'TIMING\'].append(dict(phase=\'TIMING\',root=root,gear=gid,monster=mid,\n                                tier=tier,build_id=bid,policy=policy,variant=\'CANON\',R=R_TIMING))\n    # Unselected reference grid 5 roots × 3 gears × 2 monsters × 2 tiers × 4 builds × R64\n    for root in ROOTS:\n        for gid in GEARS:\n            for mid in SPECS:\n                for tier in TIERS:\n                    for bid in (\'BASE_0\',\'OFF_T1_0\',\'OFF_T1_1\',\'OFF_T1_2\'):\n                        phases[\'OFFENSE\'].append(dict(phase=\'OFFENSE\',root=root,gear=gid,\n                            monster=mid,tier=tier,build_id=bid,policy=\'OFFENSE_ONLY\',variant=\'CANON\',R=R_OFFENSE))\n    # 1 root × 3 gears × 2 monsters × 2 tiers × 3 explicit arms × 3 policies × R128\n    for gid in GEARS:\n        for mid in SPECS:\n            for tier in TIERS:\n                for bid,variant in [(\'BASE_0\',\'CANON\'),(\'DEF_T1_2\',\'CANON\'),(\'DEF_T1_2\',\'WATER_EFF_FLAT_MINUS_2\')]:\n                    for policy in POLICIES_WATER:\n                        phases[\'WATER_QI\'].append(dict(phase=\'WATER_QI\',root=\'agua\',gear=gid,\n                            monster=mid,tier=tier,build_id=bid,policy=policy,variant=variant,R=R_WATER))\n    assert {p:len(v) for p,v in phases.items()}=={\'TIMING\':360,\'OFFENSE\':240,\'WATER_QI\':108}\n    assert sum(sum(j[\'R\'] for j in v) for v in phases.values())==75264\n    return phases\n\n\ndef compiled_qi_audit():\n    """Confirm case and bounded candidate, including Tramo II/III *only as future cost diagnostics*."""\n    eng=b.G[\'eng\'];spec=b.G[\'tech\'][\'techniques\'][\'espejo_luna\']\n    def compile_one(choices,mode):\n        sd=copy.deepcopy(spec)\n        if mode==\'WATER_EFF_FLAT_MINUS_2\':\n            sd[\'model_params\'][\'eff\'][\'t1_flat_cost\']=-2\n        c=eng.compile_technique(sd,list(choices),\'agua\')\n        return dict(cost=int(c[\'qi_cost\']),raw=float(c[\'qi_cost_raw\']),families=list(c[\'families\']),defensive=c[\'defensive\'])\n    base=compile_one((),\'CANON\')\n    canon=compile_one((2,),\'CANON\')\n    candidate=compile_one((2,),\'WATER_EFF_FLAT_MINUS_2\')\n    if not (base[\'cost\']==5 and canon[\'cost\']==5 and candidate[\'cost\']==4):\n        raise RuntimeError(f\'WATER_QI_BREAKPOINT_UNEXPECTED:{base}:{canon}:{candidate}\')\n    future=[]\n    for choices in [(2,0),(2,1),(2,2),(2,0,0),(2,1,1),(2,2,2)]:\n        a=compile_one(choices,\'CANON\');b_=compile_one(choices,\'WATER_EFF_FLAT_MINUS_2\')\n        future.append(dict(choices=list(choices),stage=\'FUTURE_NOT_AUTHORIZED_LII\',\n                           current_cost=a[\'cost\'],candidate_cost=b_[\'cost\'],\n                           current_raw=a[\'raw\'],candidate_raw=b_[\'raw\']))\n    # The earlier report misidentified OFF_T1_0 as DIRECT. Verify family truth from source.\n    families={}\n    for root in ROOTS:\n        spec_o=b.G[\'tech\'][\'techniques\'][b.OFF[root]]\n        families[root]={f\'OFF_T1_{i}\':spec_o[\'tramos\'][0][i][\'family\'] for i in range(3)}\n    if families[\'fuego\']!={\'OFF_T1_0\':\'DOT\',\'OFF_T1_1\':\'DIRECT\',\'OFF_T1_2\':\'EFFICIENCY\'}:\n        raise RuntimeError(\'FIRE_BRANCH_FAMILY_DRIFT\')\n    return {\'status\':\'PASS_PRE_REGISTERED_BREAKPOINT\',\'baseline_water\':base,\n            \'canonical_water_t1_eff\':canon,\'candidate_water_t1_eff\':candidate,\n            \'future_cost_exploration_NO_FREEZE\':future,\n            \'offensive_t1_branch_map\':families,\n            \'scope\':\'lab-only localized parameter change; no global rounding changes\'}\n\n\ndef threat_aware_policy(state,def_acts):\n    """Published cadence and current HP/Qi only; never look at future RNG or hidden intent."""\n    root=state.player_root;off=b.OFF[root];df=b.DEF[root]\n    if def_acts>=2 or state.player_defense is not None:return off\n    monster_frac=state.monster.hp/max(1.,state.monster.hp_max)\n    if monster_frac<=.38:return off\n    cdef=state.compiled[df][\'qi_cost\'];coff=state.compiled[off][\'qi_cost\']\n    if state.player.qi<cdef+2*coff:return off\n    cadence=int(state.monster_profile[\'technique\'][\'params\'][\'cadence\'])\n    # Cast a round BEFORE the next published due, or at serious current HP risk.\n    next_due=(state.round_no+1)%cadence==0\n    hp_critical=state.player.hp/max(1.,state.player.hp_max)<=.48\n    return df if (next_due or hp_critical) else off\n\n\ndef run_fight(job,replicate):\n    eng=b.G[\'eng\']\n    root=job[\'root\'];tier=job[\'tier\'];mid=job[\'monster\'];variant=job[\'variant\']\n    paths=choice_paths(root,job[\'build_id\'])\n    state=b.make_state(root,paths,job[\'gear\'],mid,tier)\n    if variant==\'WATER_EFF_FLAT_MINUS_2\':\n        if root!=\'agua\' or paths.get(\'espejo_luna\')!=[2]:\n            raise RuntimeError(\'CANDIDATE_SCOPE_DRIFT\')\n        spec=copy.deepcopy(b.G[\'tech\'][\'techniques\'][\'espejo_luna\'])\n        spec[\'model_params\'][\'eff\'][\'t1_flat_cost\']=-2\n        c=eng.compile_technique(spec,[2],\'agua\')\n        if c[\'qi_cost\']!=4:raise RuntimeError(\'WATER_QI_NOT_4\')\n        state.compiled[\'espejo_luna\']=c\n    if variant not in (\'CANON\',\'WATER_EFF_FLAT_MINUS_2\'):\n        raise RuntimeError(\'UNREGISTERED_CANDIDATE\')\n    seed=b.stable_seed(SEED_TAG,job[\'phase\'],root,job[\'gear\'],mid,tier,replicate)\n    rng=random.Random(seed)\n    reco=b.Recognition()\n    natural=anticipatory=due=miss=defs=aoe=basic=def_attempts=0\n    rounds_def_before_due=0\n    for _ in range(b.MAX_ROUNDS):\n        if not (state.player.alive() and state.monster.alive()):break\n        state.round_no+=1\n        eng.player_turn_start(state,rng)\n        if not state.player.alive():break\n        if job[\'policy\']==\'THREAT_AWARE\':\n            tid=threat_aware_policy(state,defs)\n        else:\n            tid=b.causal_policy_decision(state,job[\'policy\'],defs)\n        c=state.compiled[tid]\n        is_due_next=((state.round_no+1)%int(state.monster_profile[\'technique\'][\'params\'][\'cadence\'])==0)\n        before=float(state.monster.hp)\n        if c[\'role\']==\'DEFENSIVE\':def_attempts+=1\n        performed=eng.execute_player_technique(state,rng,c)\n        if not performed:\n            eng.execute_basic(state,rng)\n            cat=\'PLAYER_BASIC\';is_def=False;basic+=1\n        else:\n            is_def=(c[\'role\']==\'DEFENSIVE\')\n            cat=\'PLAYER_DEFENSIVE_TECHNIQUE\' if is_def else \'PLAYER_UNITARGET_TECHNIQUE\'\n        defs+=int(is_def)\n        if is_def and is_due_next:rounds_def_before_due+=1\n        aoe+=int(cat==\'PLAYER_AOE_TECHNIQUE\')\n        dealt=max(0.,before-state.monster.hp)\n        effective=\'EFECTIVA\' if (is_def or dealt>0) else \'FALLIDA\'\n        recognized=reco.observe(cat,effective) if tier==\'T2\' else None\n        if not state.monster.alive():break\n        eng.monster_turn_start(state,rng)\n        if not state.monster.alive():break\n        trigger=b.maybe_arm(state,mid,tier,recognized,dealt)\n        natural+=int(trigger==\'NATURAL\');anticipatory+=int(trigger==\'ANTICIPATORY\')\n        params=state.monster_profile[\'technique\'][\'params\']\n        is_due=state.round_no%int(params[\'cadence\'])==0 and not state.monster.skip_next_action\n        eng.execute_monster_turn(state,rng)\n        if is_due:\n            due+=1\n            if not state.monster_recent_actions or state.monster_recent_actions[-1]!=\'TECHNIQUE\':miss+=1\n        eng.end_round(state)\n    m=state.metrics\n    return dict(win=int(state.monster.hp<=0 and state.player.hp>0),\n        hp_pct=max(0.,state.player.hp)/state.player.hp_max,qi_final=max(0.,state.player.qi),\n        qi_spent=float(m.qi_spent),rounds=int(state.round_no),defense_uses=defs,\n        defense_procs=int(m.defense_procs),player_def_prevented=float(m.player_def_prevented),\n        player_absorbed=float(m.player_absorbed),player_evades=int(m.player_evades),\n        qi_drained=float(m.qi_drained),monster_damage_direct=float(m.monster_damage_direct),\n        monster_damage_dot=float(m.monster_damage_dot),player_damage_direct=float(m.player_damage_direct),\n        player_damage_dot=float(m.player_damage_dot),control_successes=int(m.control_successes),\n        natural=natural,anticipatory=anticipatory,recognitions=reco.trigger_count,\n        canonical_due=due,canonical_due_missed=miss,fallback_basics=basic,\n        aoe_uses=aoe,defense_attempts=def_attempts,pre_due_casts=rounds_def_before_due,\n        timeout=int(state.round_no>=b.MAX_ROUNDS and state.player.alive() and state.monster.alive()))\n\n\ndef worker_init(repo):\n    b.load_sources(repo)\n    b.validate_contracts()\n    if len(b.event_tests())!=10:raise RuntimeError(\'EVENT_PARITY_IN_WORKER\')\n    compiled_qi_audit()\n\n\ndef run_job(job):\n    res=[]\n    for i in range(job[\'R\']):\n        res.append({k:v for k,v in job.items() if k!=\'R\'} |\n                   {\'replicate\':i,**run_fight(job,i)})\n    return res\n\n\ndef run_checkpoints(phase,jobs,out,workers,signature):\n    cp=out/\'checkpoints\'/phase;cp.mkdir(parents=True,exist_ok=True)\n    total=sum(j[\'R\'] for j in jobs)\n    progress=b.Progress(total,\'V03 \'+phase)\n    rows=[]\n    chunk_size={\'TIMING\':15,\'OFFENSE\':16,\'WATER_QI\':9}[phase]\n    chunks=[jobs[i:i+chunk_size] for i in range(0,len(jobs),chunk_size)]\n    cached=0\n    try:\n        from contextlib import nullcontext\n        with (ProcessPoolExecutor(max_workers=workers,initializer=worker_init,initargs=(str(signature[\'repo\']),)) if workers>1 else nullcontext(None)) as pool:\n            for idx,chunk in enumerate(chunks):\n                fn=cp/f\'chunk_{idx:03d}.jsonl.gz\'\n                expected=sum(j[\'R\'] for j in chunk)\n                if fn.exists():\n                    progress.cp()\n                    with gzip.open(fn,\'rt\',encoding=\'utf8\') as f:rr=[json.loads(line) for line in f if line.strip()]\n                    if len(rr)!=expected:raise RuntimeError(f\'BAD_CHECKPOINT_ROW_COUNT:{phase}:{idx}\')\n                    def job_key(x):return (x[\'phase\'],x[\'root\'],x[\'gear\'],x[\'monster\'],x[\'tier\'],x[\'build_id\'],x[\'policy\'],x[\'variant\'])\n                    if {job_key(j) for j in chunk}!={job_key(r) for r in rr}:\n                        raise RuntimeError(f\'BAD_CHECKPOINT_KEYS:{phase}:{idx}\')\n                    if len({(*job_key(r),r[\'replicate\']) for r in rr})!=len(rr):\n                        raise RuntimeError(f\'DUPLICATE_CHECKPOINT_ROWS:{phase}:{idx}\')\n                    rows+=rr;cached+=1;progress.resume();progress.update(expected);continue\n                results=[run_job(j) for j in chunk] if pool is None else list(pool.map(run_job,chunk,chunksize=1))\n                rr=[r for rs in results for r in rs]\n                if len(rr)!=expected:raise RuntimeError(\'WORKER_OUTPUT_COUNT_MISMATCH\')\n                progress.cp()\n                tmp=Path(str(fn)+\'.tmp\')\n                with gzip.open(tmp,\'wt\',encoding=\'utf8\') as f:\n                    for r in rr:f.write(json.dumps(r,sort_keys=True)+\'\\n\')\n                tmp.replace(fn)\n                rows+=rr;progress.resume();progress.update(expected)\n        progress.done()\n    except Exception:\n        progress.err();raise\n    return rows,cached,len(chunks)\n\n\ndef paired(df,lo,hi,keys):\n    if not len(lo) or not len(hi):raise RuntimeError(\'EMPTY_COMPARISON\')\n    return b.causal_paired_summary(df,(lo,hi),keys)\n\n\ndef aggregate(df,keys):\n    return df.groupby(keys,dropna=False,as_index=False).agg(\n        R=(\'win\',\'size\'),win_rate=(\'win\',\'mean\'),hp_pct=(\'hp_pct\',\'mean\'),\n        qi_spent=(\'qi_spent\',\'mean\'),qi_final=(\'qi_final\',\'mean\'),rounds=(\'rounds\',\'mean\'),\n        defense_uses=(\'defense_uses\',\'mean\'),player_absorbed=(\'player_absorbed\',\'mean\'),\n        player_evades=(\'player_evades\',\'mean\'),natural=(\'natural\',\'mean\'),anticipatory=(\'anticipatory\',\'mean\'),\n        due_missed=(\'canonical_due_missed\',\'sum\'),pre_due_casts=(\'pre_due_casts\',\'mean\'))\n\n\ndef make_summaries(df,out):\n    timings=df[df.phase==\'TIMING\']; t=[]\n    keys=[\'root\',\'gear\',\'monster\',\'tier\',\'build_id\']\n    a=timings[timings.policy==\'OFFENSE_ONLY\']\n    for policy in POLICIES_TIMING[1:]:\n        d=paired(timings,a,timings[timings.policy==policy],keys)\n        d.insert(0,\'policy\',policy);t.append(d)\n    dt=pd.concat(t,ignore_index=True)\n    dt[\'confirmed_material_loss\']=(dt.delta_win_pp<=-5)&(dt.delta_win_ci95_high_pp<0)\n    dt.to_csv(out/\'TIMING_POLICY_PAIRED.csv.gz\',index=False,compression=\'gzip\')\n    aggregate(timings,[\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'policy\']).to_csv(out/\'TIMING_SUMMARY.csv\',index=False)\n    # Positive unique ROOT / sample: no prior selection by focal R256 results.\n    off=df[df.phase==\'OFFENSE\'];o=[]\n    keys_o=[\'root\',\'gear\',\'monster\',\'tier\']\n    baseline=off[off.build_id==\'BASE_0\']\n    for variant in (\'OFF_T1_0\',\'OFF_T1_1\',\'OFF_T1_2\'):\n        r=paired(off,baseline,off[off.build_id==variant],keys_o)\n        r.insert(0,\'variant\',variant)\n        r.insert(1,\'family\',r[\'root\'].map(lambda root:b.G[\'tech\'][\'techniques\'][b.OFF[root]][\'tramos\'][0][int(variant[-1])][\'family\']))\n        o.append(r)\n    do=pd.concat(o,ignore_index=True)\n    do.to_csv(out/\'OFFENSIVE_BRANCH_PAIRED.csv.gz\',index=False,compression=\'gzip\')\n    aggregate(off,[\'root\',\'gear\',\'monster\',\'tier\',\'build_id\']).to_csv(out/\'OFFENSIVE_SUMMARY.csv\',index=False)\n    # Water comparisons canonical baseline/eff, and identical eff with only cost candidate.\n    water=df[df.phase==\'WATER_QI\'];w=[]\n    wk=[\'gear\',\'monster\',\'tier\',\'policy\']\n    base=water[(water.build_id==\'BASE_0\')&(water.variant==\'CANON\')]\n    canon=water[(water.build_id==\'DEF_T1_2\')&(water.variant==\'CANON\')]\n    cand=water[(water.build_id==\'DEF_T1_2\')&(water.variant==\'WATER_EFF_FLAT_MINUS_2\')]\n    for title,left,right in [\n        (\'CANON_EFF_VS_BASE\',base,canon),\n        (\'CANDIDATE_MINUS2_VS_CANON_EFF\',canon,cand),\n        (\'CANDIDATE_MINUS2_VS_BASE\',base,cand),\n    ]:\n        d=paired(water,left,right,wk);d.insert(0,\'comparison\',title);w.append(d)\n    dw=pd.concat(w,ignore_index=True)\n    dw.to_csv(out/\'WATER_QI_PAIRED.csv.gz\',index=False,compression=\'gzip\')\n    aggregate(water,[\'gear\',\'monster\',\'tier\',\'build_id\',\'variant\',\'policy\']).to_csv(out/\'WATER_QI_SUMMARY.csv\',index=False)\n    # Paired check CANON eff is actually inert in all raw metric columns.\n    per=[\'gear\',\'monster\',\'tier\',\'policy\',\'replicate\']\n    a=base.set_index(per).sort_index(); bb=canon.set_index(per).sort_index()\n    if not a.index.equals(bb.index):raise RuntimeError(\'WATER_BASE_EFF_PAIRS_NOT_BALANCED\')\n    comparable=[\'win\',\'hp_pct\',\'qi_final\',\'qi_spent\',\'rounds\',\'defense_uses\',\'player_absorbed\',\'player_evades\',\'natural\',\'anticipatory\']\n    water_ineffective=bool((a[comparable].to_numpy()==bb[comparable].to_numpy()).all())\n    if not water_ineffective:raise RuntimeError(\'CANON_WATER_EFF_CHANGED_COMBAT_UNEXPECTEDLY\')\n    return dt,do,dw,{\'canonical_water_efficiency_inert\':water_ineffective,\n        \'water_base_vs_canon_pairs\':len(a),\n        \'material_timing_losses_count\':int(dt.confirmed_material_loss.sum())}\n\n\ndef main(argv=None):\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--repo\',required=True);ap.add_argument(\'--out\',required=True)\n    ap.add_argument(\'--runner\',required=True);ap.add_argument(\'--base-runner\',required=True)\n    ap.add_argument(\'--workers\',type=int,default=2)\n    args=ap.parse_args(argv)\n    repo=Path(args.repo).resolve();out=Path(args.out).resolve();out.mkdir(parents=True,exist_ok=True)\n    source,contracts,events=initial_locks(repo,args.runner,args.base_runner)\n    qi=compiled_qi_audit()\n    phases=build_jobs()\n    signatures={\'repo\':str(repo),\'commit\':source[\'commit\'],\'source_blobs\':{k:v[\'blob_sha\'] for k,v in source[\'files\'].items()},\n        \'runner_sha256\':sha256(args.runner),\'base_runner_sha256\':sha256(args.base_runner),\n        \'lab_id\':LAB_ID,\'seed_tag\':SEED_TAG,\'policy\':list(POLICIES_TIMING),\'gear\':list(GEARS),\n        \'R\':{\'timing\':R_TIMING,\'offense\':R_OFFENSE,\'water_qi\':R_WATER},\n        \'jobs_hash\':hashlib.sha256(json.dumps(phases,sort_keys=True).encode()).hexdigest()}\n    sfile=out/\'CHECKPOINT_RUN_SIGNATURE.json\'\n    if sfile.exists():\n        if json.loads(sfile.read_text())!=signatures:raise RuntimeError(\'CHECKPOINT_SIGNATURE_MISMATCH_USE_FRESH_OUTPUT_DIR\')\n    else:b.save_json(sfile,signatures)\n    b.save_json(out/\'SOURCE_LOCK.json\',source)\n    b.save_json(out/\'FROZEN_CONTRACT_PREFLIGHT.json\',contracts)\n    b.save_json(out/\'EVENT_PARITY_TESTS.json\',events)\n    b.save_json(out/\'QI_COMPILER_AND_BRANCH_IDENTITIES.json\',qi)\n    b.save_json(out/\'RUN_MANIFEST.json\',{\'lab_id\':LAB_ID,\'source_commit\':source[\'commit\'],\n        \'planned_fights\':75264,\'phases\':{p:dict(jobs=len(js),fights=sum(j[\'R\'] for j in js)) for p,js in phases.items()},\n        \'workers\':args.workers,\'candidate_only_water_efficiency\':True,\n        \'freeze\':False,\'scope\':\'No mutants, no multiroot, no LII concordances. Lab only.\'})\n    allrows=[];cached={};checkpoints={}\n    for phase,jobs in phases.items():\n        rows,c,n=run_checkpoints(phase,jobs,out,args.workers,signatures)\n        allrows+=rows;cached[phase]=c;checkpoints[phase]=n\n    df=pd.DataFrame(allrows)\n    if len(df)!=75264:raise RuntimeError(f\'REPRESENTED_FIGHTS:{len(df)}!=75264\')\n    pk=[\'phase\',\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'policy\',\'variant\',\'replicate\']\n    if df.duplicated(pk).any():raise RuntimeError(\'DUPLICATE_REPLICATE_KEY\')\n    if df.isna().any().any() or not np.isfinite(df.select_dtypes(include=\'number\').to_numpy()).all():\n        raise RuntimeError(\'NAN_OR_NONFINITE_FIGHT\')\n    for col in [\'aoe_uses\',\'canonical_due_missed\',\'timeout\']:\n        if df[col].sum():raise RuntimeError(f\'HARD_GUARD:{col}={df[col].sum()}\')\n    dt,do,dw,watercheck=make_summaries(df,out)\n    df.to_csv(out/\'RAW_SEED_RESULTS.csv.gz\',index=False,compression=\'gzip\')\n    risks=dt[dt.confirmed_material_loss].sort_values(\'delta_win_pp\')\n    risks.to_csv(out/\'TIMING_MATERIAL_RISKS.csv\',index=False)\n    offgroup=do.groupby([\'root\',\'family\',\'variant\'],as_index=False).agg(\n        contexts=(\'R\',\'size\'),mean_delta_win_pp=(\'delta_win_pp\',\'mean\'),\n        min_delta_win_pp=(\'delta_win_pp\',\'min\'),max_delta_win_pp=(\'delta_win_pp\',\'max\'),\n        mean_delta_hp_pp=(\'delta_hp_pct_pp\',\'mean\'))\n    offgroup.to_csv(out/\'OFFENSIVE_BRANCH_AGGREGATE.csv\',index=False)\n    watergroup=dw.groupby(\'comparison\',as_index=False).agg(contexts=(\'R\',\'size\'),\n        mean_delta_win_pp=(\'delta_win_pp\',\'mean\'),min_delta_win_pp=(\'delta_win_pp\',\'min\'),\n        mean_delta_hp_pp=(\'delta_hp_pct_pp\',\'mean\'),mean_delta_qi_spent=(\'delta_qi_spent\',\'mean\'))\n    watergroup.to_csv(out/\'WATER_QI_AGGREGATE.csv\',index=False)\n    decision={\n        \'status\':\'REVIEW_REQUIRED_NO_AUTOFREEZE\',\n        \'run_integrity\':\'PASS\',\'represented_fights\':len(df),\'event_tests_passed\':len(events),\n        \'unexpected_aoe\':0,\'canonical_due_missed\':0,\'timeouts\':0,\n        \'water_compiler_confirmed\':watercheck[\'canonical_water_efficiency_inert\'],\n        \'water_qi_current\':5,\'water_qi_candidate\':4,\n        \'material_timing_negative_contexts\':len(risks),\n        \'offensive_family_correction\':\'Fire OFF_T1_0=DOT; OFF_T1_1=DIRECT, not the reverse\',\n        \'candidate_edit_is_local_only\':True,\n        \'frozen_monsters_t0_t1_t2_unchanged\':True,\n        \'limitations\':[\'only normal sapo/escarabajo\',\'no mutants\',\'no multi-root\',\'no LII concordances\',\n                       \'not gameplay acquisition distribution\',\'fixed R64 for unselected offense cohorts\',\n                       \'paired seeds are CRN, different policies change action/RNG ordering\'],\n        \'next\':\'Human audit for Qi candidate and actual branch dominance; escalate only if required.\'}\n    b.save_json(out/\'DECISION_GATES.json\',decision)\n    b.save_json(out/\'CHECKPOINT_HASHES.json\',{p:sha256(p) for p in sorted((out/\'checkpoints\').rglob(\'chunk_*.jsonl.gz\'))})\n    shutil.copy2(args.runner,out/\'V03_RUNNER.py\');shutil.copy2(args.base_runner,out/\'LAB_RUNNER.py\')\n    outputs=[\'SOURCE_LOCK.json\',\'FROZEN_CONTRACT_PREFLIGHT.json\',\'EVENT_PARITY_TESTS.json\',\n        \'QI_COMPILER_AND_BRANCH_IDENTITIES.json\',\'RUN_MANIFEST.json\',\'CHECKPOINT_RUN_SIGNATURE.json\',\n        \'RAW_SEED_RESULTS.csv.gz\',\'TIMING_POLICY_PAIRED.csv.gz\',\'TIMING_SUMMARY.csv\',\n        \'OFFENSIVE_BRANCH_PAIRED.csv.gz\',\'OFFENSIVE_SUMMARY.csv\',\n        \'WATER_QI_PAIRED.csv.gz\',\'WATER_QI_SUMMARY.csv\',\n        \'TIMING_MATERIAL_RISKS.csv\',\'OFFENSIVE_BRANCH_AGGREGATE.csv\',\'WATER_QI_AGGREGATE.csv\',\n        \'DECISION_GATES.json\',\'CHECKPOINT_HASHES.json\',\'V03_RUNNER.py\',\'LAB_RUNNER.py\']\n    manifest={\'files\':{x:{\'sha256\':sha256(out/x),\'size\':(out/x).stat().st_size} for x in outputs},\n              \'total\':len(outputs),\'cached_chunks\':cached,\'checkpoints\':checkpoints}\n    b.save_json(out/\'PACKAGE_MANIFEST.json\',manifest)\n    zipout=out/f\'{LAB_ID}_REVIEW.zip\'\n    with zipfile.ZipFile(zipout,\'w\',zipfile.ZIP_DEFLATED,compresslevel=8) as z:\n        for x in [*outputs,\'PACKAGE_MANIFEST.json\']:z.write(out/x,x)\n    print(json.dumps({\'review\':str(zipout),\'review_sha256\':sha256(zipout),**decision},ensure_ascii=False,indent=2),flush=True)\n    return zipout\n\nif __name__==\'__main__\':main()\n', 'LII_V04_EXHAUSTIVE_BALANCE_GATE_RUNNER.py': '"""La Grulla Blanca: LII V04 exhaustive core, lab-only, no canonical edits.\n\nReuses frozen T0/T1/T2 adapters and legal stage II monoroot builds.\nMutant and multiroot contracts: fail-closed discovery, not fabricated.\n"""\nfrom __future__ import annotations\nimport argparse, copy, gzip, hashlib, json, math, os, random, shutil, statistics, subprocess, time, zipfile\nfrom pathlib import Path\nfrom concurrent.futures import ProcessPoolExecutor\nfrom contextlib import nullcontext\nfrom collections import defaultdict\nimport numpy as np\nimport pandas as pd\nimport LAB_RUNNER as b\nimport LII_TRAMO_I_QI_BREAKPOINT_AND_POLICY_DIAG_V03_RUNNER as v3\n\nLAB_ID=\'LII_V04_EXHAUSTIVE_BALANCE_GATE\'\nSEED_TAG=\'V04_FRESH_20261008_ROUND_ROBIN_NEW_SEEDS\'\nEXPECTED_COMMIT=\'9a6baa41bd6fc505a755a474cd4f64a2522ba0b9\'\nGEARS=(\'MANDATORY_ENTRY\',\'CARRY_OVER_LI_HIGH\',\'EXPECTED_STAGE\',\'HIGH_ROLL_STRESS\')\nTIERS=(\'T1\',\'T2\')\nPOLICIES=(\'OFFENSE_ONLY\',\'DEFENSE_OPEN\',\'DEFENSE_DUE\',\'DEFENSE_GUARD\',\'THREAT_AWARE\')\nCANDIDATES={\n  \'CANON\':{},\n  \'FIRE_DOT12\':{\'technique\':\'palma_ardiente\',\'t1_dot_pct_nominal\':0.12},\n  \'FIRE_DOT_TICKS2\':{\'technique\':\'palma_ardiente\',\'t1_dot_ticks\':2},\n  \'WATER_DEF_EFF_MINUS2\':{\'technique\':\'espejo_luna\',\'t1_flat_cost\':-2},\n  \'WATER_OFF_EFF_MINUS2\':{\'technique\':\'latigazo_marea\',\'t1_flat_cost\':-2},\n}\nR={\'GENERAL\':128,\'FIRE_DOT\':256,\'WATER_QI\':128,\'TIMING\':128}\nMETRICS=(\'win\',\'hp_pct\',\'qi_final\',\'qi_spent\',\'rounds\',\'player_absorbed\',\n         \'player_def_prevented\',\'player_damage_direct\',\'player_damage_dot\',\n         \'monster_damage_direct\',\'monster_damage_dot\',\'defense_uses\',\'player_evades\',\n         \'natural\',\'anticipatory\',\'canonical_due\',\'canonical_due_missed\',\'fallback_basics\')\nWORKER_READY=False\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for buf in iter(lambda:f.read(1<<20),b\'\'):h.update(buf)\n    return h.hexdigest()\n\ndef atomic_json(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\')\n    tmp.write_text(json.dumps(value,ensure_ascii=False,sort_keys=True,indent=2,default=str)+\'\\n\',encoding=\'utf8\')\n    os.replace(tmp,path)\n\ndef make_jobs():\n    jobs={\'GENERAL\':[],\'FIRE_DOT\':[],\'WATER_QI\':[],\'TIMING\':[]}\n    # 5 roots, 4 gear, 2 species, T1/T2, ALL 16 legal monoroot 0–2pt builds, 2 policies, R128.\n    for root in b.ROOTS:\n        for gear in GEARS:\n            for monster in b.SPECIES:\n                for tier in TIERS:\n                    for build,_,_ in b.builds_for_root(root):\n                        for pol in (\'OFFENSE_ONLY\',\'DEFENSE_GUARD\'):\n                            jobs[\'GENERAL\'].append(dict(phase=\'GENERAL\',root=root,gear=gear,monster=monster,\n                              tier=tier,build_id=build,policy=pol,variant=\'CANON\',R=R[\'GENERAL\']))\n    # Fire: 6 distinct branches (two candidates apply only to DOT).\n    fire_arms=((\'BASE_0\',\'CANON\'),(\'OFF_T1_0\',\'CANON\'),(\'OFF_T1_0\',\'FIRE_DOT12\'),\n               (\'OFF_T1_0\',\'FIRE_DOT_TICKS2\'),(\'OFF_T1_1\',\'CANON\'),(\'OFF_T1_2\',\'CANON\'))\n    for gear in GEARS:\n        for monster in b.SPECIES:\n            for tier in TIERS:\n                for build,variant in fire_arms:\n                    for pol in (\'OFFENSE_ONLY\',\'DEFENSE_GUARD\'):\n                        jobs[\'FIRE_DOT\'].append(dict(phase=\'FIRE_DOT\',root=\'fuego\',gear=gear,monster=monster,\n                          tier=tier,build_id=build,policy=pol,variant=variant,R=R[\'FIRE_DOT\']))\n    # Water 2 techniques x base/current eff/candidate, with meaningful policies.\n    water_arms=((\'OFF\',\'BASE_0\',\'CANON\'),(\'OFF\',\'OFF_T1_2\',\'CANON\'),(\'OFF\',\'OFF_T1_2\',\'WATER_OFF_EFF_MINUS2\'),\n                (\'DEF\',\'BASE_0\',\'CANON\'),(\'DEF\',\'DEF_T1_2\',\'CANON\'),(\'DEF\',\'DEF_T1_2\',\'WATER_DEF_EFF_MINUS2\'))\n    for gear in GEARS:\n        for monster in b.SPECIES:\n            for tier in TIERS:\n                for mode,build,variant in water_arms:\n                    for pol in ((\'OFFENSE_ONLY\',\'DEFENSE_GUARD\') if mode==\'OFF\' else (\'DEFENSE_OPEN\',\'DEFENSE_GUARD\')):\n                        jobs[\'WATER_QI\'].append(dict(phase=\'WATER_QI\',root=\'agua\',gear=gear,monster=monster,\n                           tier=tier,build_id=build,policy=pol,variant=variant,water_mode=mode,R=R[\'WATER_QI\']))\n    # Timing comparators: 2 roots, 4 gears, 2 native species, 2 tiers, 3 builds, 5 policies.\n    for root in (\'fuego\',\'tierra\'):\n        builds=(\'BASE_0\',\'OFF_T1_0\',\'DEF_T1_1\') if root==\'fuego\' else (\'BASE_0\',\'OFF_T1_0\',\'DEF_T1_0\')\n        for gear in GEARS:\n            for monster in b.SPECIES:\n                for tier in TIERS:\n                    for build in builds:\n                        for pol in POLICIES:\n                            jobs[\'TIMING\'].append(dict(phase=\'TIMING\',root=root,gear=gear,monster=monster,\n                                 tier=tier,build_id=build,policy=pol,variant=\'CANON\',R=R[\'TIMING\']))\n    expected={\'GENERAL\':(2560,327680),\'FIRE_DOT\':(192,49152),\n              \'WATER_QI\':(192,24576),\'TIMING\':(480,61440)}\n    assert {p:(len(js),sum(j[\'R\'] for j in js)) for p,js in jobs.items()}==expected\n    for phase,js in jobs.items():\n        for j in js:\n            if j[\'tier\'] not in TIERS or j[\'root\'] not in b.ROOTS or j[\'gear\'] not in GEARS:\n                raise RuntimeError(\'INVALID_STAGE_GRID\')\n            if j[\'build_id\'] not in [bid for bid,_,_ in b.builds_for_root(j[\'root\'])]:\n                raise RuntimeError(\'INVALID_BUILD_GRID\')\n            if j[\'variant\'] not in CANDIDATES:raise RuntimeError(\'UNKNOWN_VARIANT\')\n    return jobs\n\ndef cost_preflight():\n    eng=b.G[\'eng\'];tech=b.G[\'tech\'][\'techniques\']\n    outcomes={}\n    for tid,root in ((\'espejo_luna\',\'agua\'),(\'latigazo_marea\',\'agua\')):\n        s=tech[tid]\n        base=eng.compile_technique(s,[],\'agua\')\n        curr=eng.compile_technique(s,[2],\'agua\')\n        clone=copy.deepcopy(s);clone[\'model_params\'][\'eff\'][\'t1_flat_cost\']=-2\n        cand=eng.compile_technique(clone,[2],\'agua\')\n        assert (base[\'qi_cost\'],curr[\'qi_cost\'],cand[\'qi_cost\'])==(5,5,4),(\n            tid,base[\'qi_cost\'],curr[\'qi_cost\'],cand[\'qi_cost\'])\n        outcomes[tid]={\'base_qi\':5,\'canonical_eff_qi\':5,\'candidate_eff_qi\':4,\n                       \'base_raw\':base[\'qi_cost_raw\'],\'current_raw\':curr[\'qi_cost_raw\'],\n                       \'candidate_raw\':cand[\'qi_cost_raw\']}\n    fire=tech[\'palma_ardiente\']\n    labels={i:fire[\'tramos\'][0][i][\'family\'] for i in range(3)}\n    assert labels=={0:\'DOT\',1:\'DIRECT\',2:\'EFFICIENCY\'}\n    assert float(fire[\'model_params\'][\'dot\'][\'t1\'][\'pct_nominal\'])==0.15\n    base_dot=eng.compile_technique(fire,[0],\'fuego\')[\'dot\']\n    candidates=[]\n    for field,value in ((\'pct_nominal\',0.12),(\'ticks\',2)):\n        spec=copy.deepcopy(fire);spec[\'model_params\'][\'dot\'][\'t1\'][field]=value\n        compiled=eng.compile_technique(spec,[0],\'fuego\')\n        if compiled[\'dot\'] is None:raise RuntimeError(\'FIRE_DOT_CANDIDATE_NOT_COMPILED\')\n        assert compiled[\'dot\']!=base_dot\n        candidates.append(compiled[\'dot\'])\n    if candidates[0]==candidates[1]:raise RuntimeError(\'DOT_CANDIDATES_MECHANICALLY_IDENTICAL\')\n    assert (base_dot[\'flat_per_tick\'],base_dot[\'ticks\'])==(2,3)\n    assert (candidates[0][\'flat_per_tick\'],candidates[0][\'ticks\'])==(1,3)\n    assert (candidates[1][\'flat_per_tick\'],candidates[1][\'ticks\'])==(2,2)\n    return {\'status\':\'PASS\',\'water\':outcomes,\'fire_real_tramo_i_families\':labels,\n            \'fire_dot_candidates\':{\'CANON\':\'flat 2 x 3 ticks\',\'FIRE_DOT12\':\'flat 1 x 3 ticks\',\'FIRE_DOT_TICKS2\':\'flat 2 x 2 ticks\'},\n            \'all_candidates\':\'LAB_ONLY_NO_CANONICAL_WRITES\'}\n\ndef apply_variant(state,job,paths):\n    variant=job[\'variant\'];root=job[\'root\'];eng=b.G[\'eng\']\n    if variant==\'CANON\':return\n    cfg=CANDIDATES[variant];tid=cfg[\'technique\']\n    if tid not in paths or paths[tid]!=[2 if \'EFF\' in variant else 0]:\n        raise RuntimeError(\'VARIANT_BUILD_SCOPE:\'+variant+\':\'+str(paths))\n    if root not in (\'fuego\',\'agua\'):raise RuntimeError(\'VARIANT_ROOT_SCOPE\')\n    spec=copy.deepcopy(b.G[\'tech\'][\'techniques\'][tid])\n    if \'t1_flat_cost\' in cfg:spec[\'model_params\'][\'eff\'][\'t1_flat_cost\']=cfg[\'t1_flat_cost\']\n    elif \'t1_dot_pct_nominal\' in cfg:spec[\'model_params\'][\'dot\'][\'t1\'][\'pct_nominal\']=cfg[\'t1_dot_pct_nominal\']\n    elif \'t1_dot_ticks\' in cfg:spec[\'model_params\'][\'dot\'][\'t1\'][\'ticks\']=cfg[\'t1_dot_ticks\']\n    else:raise RuntimeError(\'UNKNOWN_CANDIDATE_OPERATION\')\n    updated=eng.compile_technique(spec,paths[tid],root)\n    current=state.compiled[tid]\n    if \'t1_flat_cost\' in cfg and not(current[\'qi_cost\']==5 and updated[\'qi_cost\']==4):\n        raise RuntimeError(\'WATER_CANDIDATE_NOT_SAVING_QI\')\n    state.compiled[tid]=updated\n\ndef choose_action(state,policy,def_acts):\n    if policy==\'THREAT_AWARE\':return v3.threat_aware_policy(state,def_acts)\n    return b.causal_policy_decision(state,policy,def_acts)\n\ndef fight_one(job,rep):\n    eng=b.G[\'eng\'];root=job[\'root\'];mid=job[\'monster\'];tier=job[\'tier\']\n    paths=v3.choice_paths(root,job[\'build_id\'])\n    state=b.make_state(root,paths,job[\'gear\'],mid,tier)\n    apply_variant(state,job,paths)\n    seed=b.stable_seed(SEED_TAG,job[\'phase\'],root,job[\'gear\'],mid,tier,rep)\n    rng=random.Random(seed);rec=b.Recognition()\n    natural=anticipatory=due=miss=defs=aoe=basics=attempts=0\n    for _ in range(b.MAX_ROUNDS):\n        if not (state.player.alive() and state.monster.alive()):break\n        state.round_no+=1;eng.player_turn_start(state,rng)\n        if not state.player.alive():break\n        tid=choose_action(state,job[\'policy\'],defs)\n        c=state.compiled[tid]\n        before=state.monster.hp\n        if c[\'role\']==\'DEFENSIVE\':attempts+=1\n        ok=eng.execute_player_technique(state,rng,c)\n        if not ok:\n            eng.execute_basic(state,rng);cat=\'PLAYER_BASIC\';is_def=False;basics+=1\n        else:\n            is_def=c[\'role\']==\'DEFENSIVE\'\n            cat=\'PLAYER_DEFENSIVE_TECHNIQUE\' if is_def else \'PLAYER_UNITARGET_TECHNIQUE\'\n        defs+=int(is_def);aoe+=int(cat==\'PLAYER_AOE_TECHNIQUE\')\n        dealt=max(0.,before-state.monster.hp)\n        effective=\'EFECTIVA\' if (is_def or dealt>0) else \'FALLIDA\'\n        recognized=rec.observe(cat,effective) if tier==\'T2\' else None\n        if not state.monster.alive():break\n        eng.monster_turn_start(state,rng)\n        if not state.monster.alive():break\n        trigger=b.maybe_arm(state,mid,tier,recognized,dealt)\n        natural+=int(trigger==\'NATURAL\');anticipatory+=int(trigger==\'ANTICIPATORY\')\n        params=state.monster_profile[\'technique\'][\'params\']\n        was_due=state.round_no%int(params[\'cadence\'])==0 and not state.monster.skip_next_action\n        eng.execute_monster_turn(state,rng)\n        if was_due:\n            due+=1\n            if not state.monster_recent_actions or state.monster_recent_actions[-1]!=\'TECHNIQUE\':miss+=1\n        eng.end_round(state)\n    m=state.metrics\n    return dict(win=int(state.monster.hp<=0 and state.player.hp>0),\n      hp_pct=max(0.,state.player.hp)/state.player.hp_max,qi_final=max(0.,state.player.qi),\n      qi_spent=float(m.qi_spent),rounds=int(state.round_no),defense_uses=defs,\n      player_absorbed=float(m.player_absorbed),player_def_prevented=float(m.player_def_prevented),\n      player_evades=int(m.player_evades),player_damage_direct=float(m.player_damage_direct),\n      player_damage_dot=float(m.player_damage_dot),monster_damage_direct=float(m.monster_damage_direct),\n      monster_damage_dot=float(m.monster_damage_dot),control_successes=int(m.control_successes),\n      natural=natural,anticipatory=anticipatory,recognitions=rec.trigger_count,\n      canonical_due=due,canonical_due_missed=miss,fallback_basics=basics,\n      defense_attempts=attempts,aoe_uses=aoe,\n      timeout=int(state.round_no>=b.MAX_ROUNDS and state.player.alive() and state.monster.alive()))\n\ndef worker_init(repo):\n    b.load_sources(repo);b.validate_contracts()\n    if len(b.event_tests())!=10:raise RuntimeError(\'T1_T2_PARITY_FAILED_IN_WORKER\')\n    cost_preflight()\n\ndef execute_job(job):\n    return [{k:v for k,v in job.items() if k!=\'R\'} | {\'replicate\':rep,**fight_one(job,rep)}\n            for rep in range(job[\'R\'])]\n\ndef job_key(j):\n    return tuple(j.get(k,\'\') for k in (\'phase\',\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'policy\',\'variant\',\'water_mode\'))\n\ndef run_phase(phase,jobs,out,workers,run_sig):\n    folder=out/\'checkpoints\'/phase;folder.mkdir(parents=True,exist_ok=True)\n    chunksize={\'GENERAL\':16,\'FIRE_DOT\':8,\'WATER_QI\':12,\'TIMING\':10,\'ESCALATION\':4}[phase]\n    chunks=[jobs[i:i+chunksize] for i in range(0,len(jobs),chunksize)]\n    total=sum(j[\'R\'] for j in jobs)\n    progress=b.Progress(total,\'V04 \'+phase)\n    allrows=[];cached=0\n    try:\n        # Reuse a single process pool. No fragile nested ProcessPool workers.\n        with (ProcessPoolExecutor(max_workers=workers,initializer=worker_init,\n                                  initargs=(str(run_sig[\'repo\']),)) if workers>1 else nullcontext(None)) as pool:\n            for ix,chunk in enumerate(chunks):\n                datafile=folder/f\'chunk_{ix:04d}.jsonl.gz\'\n                metafile=folder/f\'chunk_{ix:04d}.meta.json\'\n                expected=sum(j[\'R\'] for j in chunk)\n                chunk_sig=hashlib.sha256(json.dumps(chunk,sort_keys=True).encode()).hexdigest()\n                if datafile.exists() and metafile.exists():\n                    meta=json.loads(metafile.read_text())\n                    if meta!={\'count\':expected,\'jobs_sha256\':chunk_sig,\'data_sha256\':sha(datafile)}:\n                        raise RuntimeError(\'CHECKPOINT_INTEGRITY_MISMATCH:\'+str(datafile))\n                    with gzip.open(datafile,\'rt\',encoding=\'utf8\') as f:\n                        rr=[json.loads(x) for x in f if x.strip()]\n                    if len(rr)!=expected:raise RuntimeError(\'CHECKPOINT_INCOMPLETE\')\n                    if {job_key(j) for j in chunk}!={job_key(r) for r in rr}:\n                        raise RuntimeError(\'CHECKPOINT_DIFFERENT_JOBS\')\n                    if len({(*job_key(x),x[\'replicate\']) for x in rr})!=expected:\n                        raise RuntimeError(\'CHECKPOINT_DUPLICATE_ROWS\')\n                    cached+=1;allrows.extend(rr);progress.update(expected);continue\n                if datafile.exists() or metafile.exists():\n                    raise RuntimeError(\'PARTIAL_CHECKPOINT_FILE:\'+str(datafile))\n                rr=[r for batch in ([execute_job(j) for j in chunk] if pool is None else pool.map(execute_job,chunk,chunksize=1)) for r in batch]\n                if len(rr)!=expected:raise RuntimeError(\'WORKER_RESULT_COUNT\')\n                progress.cp()\n                tmp=datafile.with_suffix(datafile.suffix+\'.tmp\')\n                with gzip.open(tmp,\'wt\',encoding=\'utf8\',compresslevel=5) as f:\n                    for r in rr:f.write(json.dumps(r,sort_keys=True,allow_nan=False)+\'\\n\')\n                os.replace(tmp,datafile)\n                atomic_json(metafile,{\'count\':expected,\'jobs_sha256\':chunk_sig,\'data_sha256\':sha(datafile)})\n                allrows.extend(rr);progress.resume();progress.update(expected)\n        progress.done()\n    except BaseException:\n        progress.err();raise\n    return allrows,{\'cached_chunks\':cached,\'total_chunks\':len(chunks),\'fights\':len(allrows)}\n\ndef paired_summary(df,lo,hi,keys,label):\n    a=df.query(\'arm == @lo\').set_index(keys+[\'replicate\']).sort_index()\n    z=df.query(\'arm == @hi\').set_index(keys+[\'replicate\']).sort_index()\n    if len(a)!=len(z) or len(a)==0 or not a.index.equals(z.index):\n        raise RuntimeError(\'UNPAIRED:\'+label+\':\'+str((len(a),len(z))))\n    per=pd.DataFrame(index=a.index)\n    for metric in (\'win\',\'hp_pct\',\'qi_spent\',\'qi_final\',\'rounds\',\'player_damage_direct\',\'player_damage_dot\',\n                   \'player_absorbed\',\'defense_uses\',\'natural\',\'anticipatory\'):\n        per[\'d_\'+metric]=z[metric].to_numpy()-a[metric].to_numpy()\n    groups=per.groupby(level=keys,sort=False)\n    out=[]\n    for group,tab in groups:\n        if not isinstance(group,tuple):group=(group,)\n        ds=tab[\'d_win\'].to_numpy(dtype=float);n=len(ds)\n        mean=float(ds.mean());se=float(ds.std(ddof=1)/np.sqrt(n)) if n>1 else float(\'inf\')\n        x=dict(zip(keys,group));x.update(comparison=label,R=n,delta_win_pp=100*mean,\n                 delta_win_ci95_low_pp=100*(mean-1.96*se),delta_win_ci95_high_pp=100*(mean+1.96*se),\n                 discordant_win_gain=int((ds>0).sum()),discordant_win_loss=int((ds<0).sum()))\n        for col in per.columns:\n            if col!=\'d_win\':x[\'delta_\'+col[2:]]=float(tab[col].mean())\n        out.append(x)\n    return pd.DataFrame(out)\n\ndef summaries(df,out):\n    # Uniform contexts, not encountered frequency in actual game.\n    rows=[];tables=[]\n    def compare(phase,baseline,candidates,keys):\n        d=df[df.phase==phase].copy();d[\'arm\']=d[\'build_id\']+\'|\'+d[\'variant\']+\'|\'+d[\'policy\']\n        for bid,variant,policy in candidates:\n            hi=\'|\'.join((bid,variant,policy));lo=\'|\'.join(baseline)\n            tab=paired_summary(d,lo,hi,keys,phase+\':\'+hi)\n            tab[\'phase\']=phase;tab[\'baseline\']=lo;tab[\'candidate\']=hi\n            tables.append(tab)\n    # GENERAL: all 16 stage-legal monoroot builds and 0/1/2 pt contrasts.\n    for p in (\'OFFENSE_ONLY\',\'DEFENSE_GUARD\'):\n        all_nonbase=[(bid,\'CANON\',p) for bid,_,_ in b.builds_for_root(\'fuego\') if bid!=\'BASE_0\']\n        compare(\'GENERAL\',(\'BASE_0\',\'CANON\',p),all_nonbase,[\'root\',\'gear\',\'monster\',\'tier\'])\n        for i in range(3):\n            for j in range(3):\n                target=(f\'OFF_{i}_DEF_{j}\',\'CANON\',p)\n                compare(\'GENERAL\',(f\'OFF_T1_{i}\',\'CANON\',p),[target],[\'root\',\'gear\',\'monster\',\'tier\'])\n                compare(\'GENERAL\',(f\'DEF_T1_{j}\',\'CANON\',p),[target],[\'root\',\'gear\',\'monster\',\'tier\'])\n    # FIRE: compare 5 arms against no investment and dot candidate vs canonical in same cohort.\n    for p in (\'OFFENSE_ONLY\',\'DEFENSE_GUARD\'):\n        arms=[(\'OFF_T1_0\',\'CANON\',p),(\'OFF_T1_0\',\'FIRE_DOT12\',p),(\'OFF_T1_0\',\'FIRE_DOT_TICKS2\',p),\n              (\'OFF_T1_1\',\'CANON\',p),(\'OFF_T1_2\',\'CANON\',p)]\n        compare(\'FIRE_DOT\',(\'BASE_0\',\'CANON\',p),arms,[\'root\',\'gear\',\'monster\',\'tier\'])\n        compare(\'FIRE_DOT\',(\'OFF_T1_0\',\'CANON\',p),arms[1:3],[\'root\',\'gear\',\'monster\',\'tier\'])\n    # WATER only same intervention mode; label in the keys to disambiguate the repeated BASE.\n    for mode,ps,bid in [(\'OFF\',(\'OFFENSE_ONLY\',\'DEFENSE_GUARD\'),\'OFF_T1_2\'),\n                        (\'DEF\',(\'DEFENSE_OPEN\',\'DEFENSE_GUARD\'),\'DEF_T1_2\')]:\n        for p in ps:\n            d=df[(df.phase==\'WATER_QI\')&(df.water_mode==mode)].copy()\n            d[\'arm\']=d[\'build_id\']+\'|\'+d[\'variant\']+\'|\'+d[\'policy\']\n            cand=f\'WATER_{mode}_EFF_MINUS2\'\n            for lo,hi in [(f\'BASE_0|CANON|{p}\',f\'{bid}|CANON|{p}\'),\n                          (f\'{bid}|CANON|{p}\',f\'{bid}|{cand}|{p}\')]:\n                tab=paired_summary(d,lo,hi,[\'root\',\'gear\',\'monster\',\'tier\'],\'WATER_QI:\'+mode+\':\'+p+\':\'+lo+\':\'+hi)\n                tab[\'phase\']=\'WATER_QI\';tab[\'water_mode\']=mode;tab[\'baseline\']=lo;tab[\'candidate\']=hi\n                tables.append(tab)\n    # TIMING: each defense policy to same build offense-only baseline.\n    for pol in POLICIES[1:]:\n        d=df[df.phase==\'TIMING\'].copy();d[\'arm\']=d[\'build_id\']+\'|\'+d[\'variant\']+\'|\'+d[\'policy\']\n        for bid in (\'BASE_0\',\'OFF_T1_0\'):\n            tab=paired_summary(d,f\'{bid}|CANON|OFFENSE_ONLY\',f\'{bid}|CANON|{pol}\',\n                 [\'root\',\'gear\',\'monster\',\'tier\'],\'TIMING:\'+bid+\':\'+pol)\n            tab[\'phase\']=\'TIMING\';tab[\'baseline\']=f\'{bid}|CANON|OFFENSE_ONLY\';tab[\'candidate\']=f\'{bid}|CANON|{pol}\'\n            tables.append(tab)\n        # For each root, defensive T1 is different valid build ID.\n        for root in (\'fuego\',\'tierra\'):\n            sub=d[d.root==root].copy()\n            bid=\'DEF_T1_1\' if root==\'fuego\' else \'DEF_T1_0\'\n            tab=paired_summary(sub,f\'{bid}|CANON|OFFENSE_ONLY\',f\'{bid}|CANON|{pol}\',\n                 [\'root\',\'gear\',\'monster\',\'tier\'],\'TIMING:\'+bid+\':\'+pol)\n            tab[\'phase\']=\'TIMING\';tab[\'baseline\']=f\'{bid}|CANON|OFFENSE_ONLY\';tab[\'candidate\']=f\'{bid}|CANON|{pol}\'\n            tables.append(tab)\n    combined=pd.concat(tables,ignore_index=True)\n    combined.to_csv(out/\'PAIRED_CONTRASTS.csv.gz\',index=False,compression=\'gzip\')\n    group=df.groupby([\'phase\',\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'variant\',\'policy\',\'water_mode\'],dropna=False,as_index=False).agg(\n        R=(\'win\',\'size\'),wins=(\'win\',\'sum\'),win_rate=(\'win\',\'mean\'),hp_pct=(\'hp_pct\',\'mean\'),\n        qi_spent=(\'qi_spent\',\'mean\'),rounds=(\'rounds\',\'mean\'),\n        player_damage_dot=(\'player_damage_dot\',\'mean\'),defense_uses=(\'defense_uses\',\'mean\'),\n        natural=(\'natural\',\'mean\'),anticipatory=(\'anticipatory\',\'mean\'),\n        missed=(\'canonical_due_missed\',\'sum\'))\n    group.to_csv(out/\'CONTEXT_SUMMARY.csv.gz\',index=False,compression=\'gzip\')\n    # Significant contexts use descriptive CI; multiple comparisons & selected cohorts require review.\n    suspicious=combined[(combined.delta_win_pp<=-5)&(combined.delta_win_ci95_high_pp<0)].copy()\n    suspicious.to_csv(out/\'RISK_CONTEXTS.csv\',index=False)\n    over=combined[(combined.delta_win_pp>=8)&(combined.delta_win_ci95_low_pp>0)].copy()\n    over.to_csv(out/\'POSITIVE_CONTEXTS.csv\',index=False)\n    ag=combined.groupby([\'phase\',\'baseline\',\'candidate\'],dropna=False,as_index=False).agg(\n        contexts=(\'R\',\'size\'),mean_delta_win_pp=(\'delta_win_pp\',\'mean\'),\n        min_delta_win_pp=(\'delta_win_pp\',\'min\'),max_delta_win_pp=(\'delta_win_pp\',\'max\'),\n        mean_delta_qi_spent=(\'delta_qi_spent\',\'mean\'),mean_delta_player_damage_dot=(\'delta_player_damage_dot\',\'mean\'))\n    ag.to_csv(out/\'PAIRED_AGGREGATE.csv\',index=False)\n    return {\'pair_rows\':len(combined),\'potential_material_losses\':len(suspicious),\n            \'potential_material_gains\':len(over),\'summary_rows\':len(group)}\n\ndef choose_escalations(contrasts,source_jobs,limit=16):\n    """Independent confirmation sample: select suspicious/high-leverage pairs.\n\n    Every escalation replays both arms with brand new R512 seeds. The screen\n    itself is NOT the confirmatory sample.\n    """\n    x=contrasts.copy()\n    negative=x[(x.delta_win_pp<=-5)&(x.delta_win_ci95_high_pp<0)].copy()\n    negative=negative.sort_values(\'delta_win_pp\')\n    high=x[(x.phase==\'FIRE_DOT\')&(x.delta_win_pp>=8)&(x.delta_win_ci95_low_pp>0)].copy()\n    high=high.sort_values(\'delta_win_pp\',ascending=False)\n    water=x[(x.phase==\'WATER_QI\')&(x[\'candidate\'].astype(str).str.contains(\'EFF_MINUS2\'))].copy()\n    water=water.sort_values(\'delta_qi_spent\')\n    pick=pd.concat([negative.head(8),high.head(4),water.head(4)],ignore_index=True)\n    # If no strong flags, use high-information checks from each phase.\n    if len(pick)<8:\n        others=x.reindex(x.delta_win_pp.abs().sort_values(ascending=False).index).head(8-len(pick))\n        pick=pd.concat([pick,others],ignore_index=True)\n    pick=pick.drop_duplicates(subset=[\'phase\',\'root\',\'gear\',\'monster\',\'tier\',\'baseline\',\'candidate\']).head(limit)\n    out=[];specs=[]\n    for row in pick.to_dict(\'records\'):\n        origin=row[\'phase\'];bid=row[\'baseline\'];cid=row[\'candidate\']\n        pairid=hashlib.sha256(json.dumps([origin,row[\'root\'],row[\'gear\'],row[\'monster\'],row[\'tier\'],bid,cid],sort_keys=True).encode()).hexdigest()[:14]\n        selected=[]\n        for arm in (bid,cid):\n            name,variant,policy=arm.split(\'|\')\n            options=[j for j in source_jobs[origin] if j[\'root\']==row[\'root\'] and j[\'gear\']==row[\'gear\']\n                and j[\'monster\']==row[\'monster\'] and j[\'tier\']==row[\'tier\'] and j[\'build_id\']==name\n                and j[\'variant\']==variant and j[\'policy\']==policy\n                and (origin!=\'WATER_QI\' or j.get(\'water_mode\')==row.get(\'water_mode\'))]\n            if len(options)!=1:\n                raise RuntimeError(\'ESCALATION_ARM_LOOKUP:\'+str((pairid,arm,len(options))))\n            y=copy.deepcopy(options[0]);y.update(phase=\'ESCALATION\',origin_phase=origin,escalation_id=pairid,R=512)\n            selected.append(y);out.append(y)\n        specs.append({\'escalation_id\':pairid,\'origin_phase\':origin,\'root\':row[\'root\'],\n          \'gear\':row[\'gear\'],\'monster\':row[\'monster\'],\'tier\':row[\'tier\'],\n          \'baseline\':bid,\'candidate\':cid,\'screen_delta_win_pp\':float(row[\'delta_win_pp\']),\n          \'screen_ci95_low_pp\':float(row[\'delta_win_ci95_low_pp\']),\n          \'screen_ci95_high_pp\':float(row[\'delta_win_ci95_high_pp\'])})\n    return out,specs\n\ndef summarize_escalations(df,specs,out):\n    if df.empty:\n        pd.DataFrame().to_csv(out/\'ESCALATION_CONTRASTS.csv\',index=False)\n        return []\n    x=df.copy();x[\'arm\']=x[\'build_id\']+\'|\'+x[\'variant\']+\'|\'+x[\'policy\']\n    summary=[]\n    for spec in specs:\n        d=x[x.escalation_id==spec[\'escalation_id\']]\n        result=paired_summary(d,spec[\'baseline\'],spec[\'candidate\'],[\'escalation_id\'],\n                              \'ESCALATION:R512:\'+spec[\'origin_phase\'])\n        if len(result)!=1 or result.R.iloc[0]!=512:raise RuntimeError(\'ESCALATION_INCOMPLETE\')\n        summary.append({**spec,**result.to_dict(\'records\')[0]})\n    z=pd.DataFrame(summary)\n    z.to_csv(out/\'ESCALATION_CONTRASTS.csv\',index=False)\n    return summary\n\ndef discover_mutants(repo):\n    """Contract provenance scan only; never invent mutant modifiers.\n\n    The mutator API/schema must be audited separately; this gate does not claim\n    mutant coverage or generalize raw win percentages to mutant populations.\n    """\n    repo=Path(repo)\n    cmd=[\'git\',\'-C\',str(repo),\'ls-files\']\n    files=subprocess.check_output(cmd,text=True).splitlines()\n    candidates=[p for p in files if any(w in p.lower() for w in (\'mutant\',\'sufij\',\'varianc\',\'mutaci\'))]\n    candidates=sorted(candidates)\n    entries=[]\n    for rel in candidates:\n        p=repo/rel\n        if not p.is_file():continue\n        blob=p.read_bytes()\n        entries.append({\'path\':rel,\'sha256\':hashlib.sha256(blob).hexdigest(),\'size\':len(blob),\n                        \'authority_marker_filename\':any(x in p.name.upper() for x in (\'MUTANT_SUFFIX_CONTRACT\',\'VARIANCE\'))})\n    contract_candidates=[e[\'path\'] for e in entries if \'MUTANT_SUFFIX_CONTRACT_V1\' in e[\'path\'].upper()]\n    variance_candidates=[e[\'path\'] for e in entries if \'VARIANCE\' in e[\'path\'].upper()]\n    return {\'status\':\'DISCOVERY_ONLY_NOT_EXECUTED\',\'paths\':entries,\n            \'mutant_contract_candidates\':contract_candidates,\'individual_variance_candidates\':variance_candidates,\n            \'mutant_fights_executed\':0,\'multiroot_fights_executed\':0,\n            \'blocker\':\'Canonical mutant/suffix/individual variance adapter and legal multi-element acquisition mapping need separate source-verified contract gate.\',\n            \'no_fake_mutant_results\':True}\n\ndef main(argv=None):\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--repo\',required=True);ap.add_argument(\'--out\',required=True)\n    ap.add_argument(\'--runner\',required=True);ap.add_argument(\'--v03-runner\',required=True)\n    ap.add_argument(\'--base-runner\',required=True);ap.add_argument(\'--workers\',type=int,default=2)\n    args=ap.parse_args(argv)\n    repo=Path(args.repo).resolve();out=Path(args.out).resolve();out.mkdir(parents=True,exist_ok=True)\n    if args.workers not in (1,2):raise RuntimeError(\'WORKERS_ALLOWED_1_OR_2\')\n    source,contract,events=v3.initial_locks(repo,args.runner,args.base_runner)\n    if source[\'commit\']!=EXPECTED_COMMIT:raise RuntimeError(\'SOURCE_COMMIT_DRIFT\')\n    if len(events)!=10 or not all(x==\'PASS\' for x in events.values()):raise RuntimeError(\'FROZEN_EVENT_PARITY_FAIL\')\n    qi=cost_preflight();jobs=make_jobs();mutants=discover_mutants(repo)\n    signature={\'repo\':str(repo),\'head\':source[\'commit\'],\n      \'source_sha256\':{k:str(v[\'sha256\']) for k,v in source[\'files\'].items()},\n      \'runners_sha256\':{\'base\':sha(args.base_runner),\'v03\':sha(args.v03_runner),\'v04\':sha(args.runner)},\n      \'lab_id\':LAB_ID,\'seed_tag\':SEED_TAG,\'R\':R,\n      \'jobs_sha256\':hashlib.sha256(json.dumps(jobs,sort_keys=True).encode()).hexdigest()}\n    # Only string dict keys, preventing the earlier PosixPath JSON crash.\n    file_sig=out/\'CHECKPOINT_RUN_SIGNATURE.json\'\n    if file_sig.exists():\n        if json.loads(file_sig.read_text())!=signature:\n            raise RuntimeError(\'CHECKPOINT_SIGNATURE_MISMATCH_USE_NEW_OUT_DIRECTORY\')\n    else:atomic_json(file_sig,signature)\n    atomic_json(out/\'SOURCE_LOCK.json\',source)\n    atomic_json(out/\'CONTRACT_PARITY.json\',{\'contract\':contract,\'events\':events,\'qi\':qi})\n    atomic_json(out/\'MUTANT_AUTHORITY_DISCOVERY.json\',mutants)\n    planned={k:{\'jobs\':len(v),\'fights\':sum(j[\'R\'] for j in v)} for k,v in jobs.items()}\n    atomic_json(out/\'RUN_MANIFEST.json\',{\'lab_id\':LAB_ID,\'source_commit\':source[\'commit\'],\n      \'phases\':planned,\'planned_normal_fights\':sum(x[\'fights\'] for x in planned.values()),\n      \'workers\':args.workers,\'mutant_fights\':0,\'multiroot_fights\':0,\n      \'auto_freeze\':False,\'candidate_changes\':\'IN_MEMORY_ONLY\'})\n    allrows=[];checkpoint_status={}\n    for phase,js in jobs.items():\n        rows,status=run_phase(phase,js,out,args.workers,signature)\n        allrows.extend(rows);checkpoint_status[phase]=status\n    df=pd.DataFrame(allrows)\n    count=sum(sum(x[\'R\'] for x in js) for js in jobs.values())\n    if len(df)!=count:raise RuntimeError(\'MISSING_RAW_FIGHTS\')\n    keys=[\'phase\',\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'variant\',\'policy\',\'replicate\']\n    if df.duplicated(keys).any():\n        # WATER_QI contains BASE_0 twice, intentionally in distinct intervention modes.\n        keys.insert(1,\'water_mode\')\n        if df.duplicated(keys).any():raise RuntimeError(\'DUPLICATE_SEED_ROWS\')\n    if df.isna(subset=[\'win\',\'hp_pct\',\'qi_spent\',\'rounds\']).any().any():raise RuntimeError(\'NAN_ROWS\')\n    if not np.isfinite(df.select_dtypes(\'number\').to_numpy()).all():raise RuntimeError(\'NONFINITE_RESULTS\')\n    for k in (\'canonical_due_missed\',\'aoe_uses\',\'timeout\'):\n        if df[k].sum():raise RuntimeError(f\'HARD_GUARD_{k}:{df[k].sum()}\')\n    df.to_csv(out/\'RAW_SEED_RESULTS.csv.gz\',index=False,compression=\'gzip\')\n    metrics=summaries(df,out)\n    # Independent R512 confirmation from the V04 screen. Never reuse seeds.\n    contrasts=pd.read_csv(out/\'PAIRED_CONTRASTS.csv.gz\')\n    escalation_jobs,escalation_specs=choose_escalations(contrasts,jobs)\n    atomic_json(out/\'ESCALATION_SELECTION.json\',escalation_specs)\n    escalation_signature=hashlib.sha256(json.dumps(escalation_jobs,sort_keys=True).encode()).hexdigest()\n    stored=out/\'ESCALATION_RUN_SIGNATURE.json\'\n    if stored.exists():\n        if json.loads(stored.read_text()) != {\'jobs_sha256\':escalation_signature}:\n            raise RuntimeError(\'ESCALATION_CHECKPOINT_SIGNATURE_MISMATCH\')\n    else:atomic_json(stored,{\'jobs_sha256\':escalation_signature})\n    erows,estatus=run_phase(\'ESCALATION\',escalation_jobs,out,args.workers,signature) if escalation_jobs else ([],{\'cached_chunks\':0,\'total_chunks\':0,\'fights\':0})\n    edf=pd.DataFrame(erows)\n    if len(edf):\n        if edf[[\'aoe_uses\',\'canonical_due_missed\',\'timeout\']].sum().sum():\n            raise RuntimeError(\'ESCALATION_HARD_GATE_FAILURE\')\n        edf.to_csv(out/\'ESCALATION_RAW_SEEDS.csv.gz\',index=False,compression=\'gzip\')\n    else:\n        pd.DataFrame().to_csv(out/\'ESCALATION_RAW_SEEDS.csv.gz\',index=False,compression=\'gzip\')\n    escalated=summarize_escalations(edf,escalation_specs,out)\n    checkpoint_status[\'ESCALATION\']=estatus\n    cps={str(p.relative_to(out)):sha(p) for p in sorted((out/\'checkpoints\').rglob(\'chunk_*.jsonl.gz\'))}\n    atomic_json(out/\'CHECKPOINT_HASHES.json\',cps)\n    decision={\'status\':\'REVIEW_REQUIRED_NO_AUTO_FREEZE\',\'run_integrity\':\'PASS\',\n      \'represented_normal_fights\':count+len(erows),\'screen_fights\':count,\'escalation_fights\':len(erows),\'normal_contexts\':sum(len(v) for v in jobs.values()),\n      \'event_tests_passed\':10,\'qa_gates\':{\'no_AOE\':True,\'no_due_missed\':True,\'no_timeouts\':True},\n      \'cost_breakpoint\':qi[\'water\'],\'candidate_changes_canonical\':False,\n      \'mutants_covered\':False,\'mutant_contract_discovery\':mutants[\'mutant_contract_candidates\'],\n      \'multi_element_builds_covered\':False,\'coverage_blockers\':[mutants[\'blocker\']],\n      \'metrics\':metrics,\'escalation_contexts\':len(escalation_specs),\n      \'escalation_95CI_negative_contexts\':sum(1 for e in escalated if e[\'delta_win_ci95_high_pp\']<0),\n      \'seed_tag\':SEED_TAG,\'statistical_limits\':[\n        \'Uniform context sampling is not in-game encounter frequency.\',\n        \'Paired seeds use common random numbers; policy differences change action and RNG order.\',\n        \'Per-context normal CI95 are exploratory and not multiple-test corrected.\',\n        \'No mutant/multi-root/concordance LII freeze from this V04 core.\'\n      ]}\n    atomic_json(out/\'DECISION_GATES.json\',decision)\n    copies={\'V04_RUNNER.py\':args.runner,\'V03_RUNNER.py\':args.v03_runner,\'LAB_RUNNER.py\':args.base_runner}\n    for name,path in copies.items():shutil.copy2(path,out/name)\n    outputs=[\'SOURCE_LOCK.json\',\'CONTRACT_PARITY.json\',\'MUTANT_AUTHORITY_DISCOVERY.json\',\n       \'RUN_MANIFEST.json\',\'CHECKPOINT_RUN_SIGNATURE.json\',\'RAW_SEED_RESULTS.csv.gz\',\n       \'PAIRED_CONTRASTS.csv.gz\',\'CONTEXT_SUMMARY.csv.gz\',\'PAIRED_AGGREGATE.csv\',\n       \'RISK_CONTEXTS.csv\',\'POSITIVE_CONTEXTS.csv\',\'ESCALATION_SELECTION.json\',\n       \'ESCALATION_RUN_SIGNATURE.json\',\'ESCALATION_RAW_SEEDS.csv.gz\',\'ESCALATION_CONTRASTS.csv\',\n       \'CHECKPOINT_HASHES.json\',\n       \'DECISION_GATES.json\',*copies]\n    manifest={\'files\':{name:{\'size\':(out/name).stat().st_size,\'sha256\':sha(out/name)} for name in outputs},\n      \'checkpoints\':checkpoint_status,\'total\':len(outputs)}\n    atomic_json(out/\'PACKAGE_MANIFEST.json\',manifest)\n    zip_path=out/(LAB_ID+\'_REVIEW.zip\')\n    tmp=zip_path.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:\n        for name in [*outputs,\'PACKAGE_MANIFEST.json\']:z.write(out/name,arcname=name)\n    os.replace(tmp,zip_path)\n    print(json.dumps({\'review_zip\':str(zip_path),\'sha256\':sha(zip_path),\n          \'decision\':decision,\'checkpoint_status\':checkpoint_status},ensure_ascii=False,indent=2),flush=True)\n    return zip_path\n\nif __name__==\'__main__\':main()\n', 'LII_V05_VARIANCE_STRESS_CROSSCHECK_RUNNER.py': '"""LII V05A — source-locked historical-envelope sensitivity and mutant authority.\n\nIMPORTANT: LII T0 variance collapsed. HIST_* is NOT a canonical mutant and\nsuffix abilities must NEVER be silently attached to it. No LII mutant balance\nfreeze is possible from this run. Includes actual LI mutant spawn contract QA.\n"""\nfrom __future__ import annotations\nimport argparse, copy, gzip, hashlib, importlib, json, math, os, random, shutil, sys, zipfile\nfrom pathlib import Path\nfrom collections import defaultdict\nfrom concurrent.futures import ProcessPoolExecutor\nfrom contextlib import nullcontext\nimport numpy as np\nimport pandas as pd\nimport LAB_RUNNER as b\nimport LII_TRAMO_I_QI_BREAKPOINT_AND_POLICY_DIAG_V03_RUNNER as v3\nimport LII_V04_EXHAUSTIVE_BALANCE_GATE_RUNNER as v4\n\nID=\'LII_V05A_MUTANT_AUTHORITY_AND_HISTORIC_ENVELOPE_CROSSCHECK\'\nPIN=\'9a6baa41bd6fc505a755a474cd4f64a2522ba0b9\'\nTAG=\'V05A_FRESH_SEEDS_20261008\'\nEXTRA={\n \'experimentos/balance_nuevo/individual_variance_v0.1/monster_individual_variance_v0.1.json\':\'b96e259284e961cde404183618510acd498314ed\',\n \'experimentos/balance_nuevo/individual_variance_v0.1/individual_variance.py\':\'9d0a8b4c0720d263dfeee89d8116c6316cabb34b\',\n \'experimentos/balance_nuevo/individual_variance_v0.1/suffix_lab_v0.1/MUTANT_SUFFIX_CONTRACT_V1.json\':\'f749aa36d73287ad868215df5f7043d2c2d5e955\',\n \'experimentos/balance_nuevo/adaptive_six_revalidation_v0.1/SIX_REPEATABLE_T0_VARIANCE_AUTHORITY_V01.json\':\'d2dda588b9a578edfb5fe63aabe0bf338637a709\'\n}\nMODES=(\'FROZEN_T0\',\'HIST_EVASION\',\'HIST_DEFENSE\',\'HIST_ATTACK\',\'HIST_JOINT_MAX\')\nBUILDS=(\'BASE_0\',\'OFF_T1_0\',\'OFF_T1_1\',\'OFF_T1_2\',\'DEF_T1_0\',\'DEF_T1_1\',\'DEF_T1_2\',\'OFF_0_DEF_0\')\nPOLICIES=(\'OFFENSE_ONLY\',\'DEFENSE_GUARD\')\nROOTS=b.ROOTS; GEARS=v4.GEARS; MONSTERS=b.SPECIES;TIERS=(\'T1\',\'T2\')\nR=32; SPAWNS_PER_LI_SPECIES=20000\nSENSITIVE={\'sapo_ceniza\',\'escarabajo_hierro\'}\nG={}\n\ndef sha(p):return v4.sha(p)\ndef blob(data):return hashlib.sha1(b\'blob \'+str(len(data)).encode()+b\'\\0\'+data).hexdigest()\ndef save(p,obj):v4.atomic_json(p,obj)\n\ndef verify_extra(repo):\n    rows={}\n    for rel,expected in EXTRA.items():\n        p=Path(repo)/rel\n        if not p.exists():raise RuntimeError(\'MISSING_AUTHORITY:\'+rel)\n        got=blob(p.read_bytes())\n        if got!=expected:raise RuntimeError(f\'AUTHORITY_BLOB_DRIFT:{rel}:{got}:{expected}\')\n        rows[rel]={\'blob_sha\':got,\'sha256\':sha(p),\'size\':p.stat().st_size}\n    return rows\n\ndef preflight(repo):\n    verify=verify_extra(repo)\n    variance=json.loads((Path(repo)/next(k for k in EXTRA if k.endswith(\'monster_individual_variance_v0.1.json\'))).read_text())\n    suffix=json.loads((Path(repo)/next(k for k in EXTRA if k.endswith(\'MUTANT_SUFFIX_CONTRACT_V1.json\'))).read_text())\n    six=json.loads((Path(repo)/next(k for k in EXTRA if k.endswith(\'SIX_REPEATABLE_T0_VARIANCE_AUTHORITY_V01.json\'))).read_text())\n    assert variance[\'schema_version\']==\'monster-individual-variance-v0.4\'\n    assert suffix[\'status\']==\'HUMAN_RATIFIED_FROZEN_FOR_T2_LAB\'\n    assert suffix[\'semantics\'][\'suffix_rng\'] is False\n    assert six[\'status\']==\'LII_T2_RECOGNITION_HUMAN_RATIFIED_2026_10_07\'\n    for mid in MONSTERS:\n        profile=six[\'species\'][mid]; v=profile[\'variance\']\n        assert v[\'status\']==\'ORDINARY_VARIANCE_COLLAPSED_TO_FLOOR_THROUGH_T2_VALIDATION\',mid\n        assert all(v[\'base\'][key]==v[\'upper_envelope\'][key] for key in (\'hp\',\'defense\',\'evasion\',\'precision\',\'tenacity\')),(mid,\'LII FROZEN ORDINARY VARIANCE NOT COLLAPSED\')\n        assert all(len(x[\'ladder\'])==1 for x in v[\'attacks\'].values()),(mid,\'LII FROZEN OFFENSIVE VARIANCE NOT COLLAPSED\')\n        t0=b.G[\'freeze\'][\'T0\'][\'profiles\'][mid]\n        for k in (\'hp\',\'defense\',\'precision\',\'evasion\',\'tenacity\'):\n            assert v[\'base\'][k]==t0[\'stats\'][k]\n        hist=v[\'historical_upper_evidence\']\n        for k in (\'hp\',\'defense\',\'precision\',\'evasion\',\'tenacity\'):\n            assert hist[\'upper_envelope\'][k] is not None\n        assert suffix[\'semantics\'][\'no_fixed_normal_monster\'] and suffix[\'semantics\'][\'roll_once_per_instance\']\n    for tid in (\'palma_ardiente\',\'latigazo_marea\',\'espejo_luna\'):\n        assert tid in b.G[\'tech\'][\'techniques\']\n    for root in ROOTS:\n        legal={x[0] for x in b.builds_for_root(root)}\n        assert set(BUILDS)<=legal,(root,set(BUILDS)-legal)\n    G.clear();G.update(variance=variance,suffix=suffix,six=six)\n    return {\'authority_sha\':verify,\'contract_status\':\'PASS\',\n      \'lii_mutant_spawn_possible_under_frozen_ordinary_variance\':False,\n      \'lii_historical_upper_envelope\':\'DIAGNOSTIC_ONLY_NOT_CANONICAL\',\n      \'suffix_combat_abilities_applied_to_lII_historical_scenarios\':False,\n      \'legal_multiroot_concordance_executed\':False,\n      \'root_graft_legality\':\'PENDING_EXPLICIT_STAGE_AUTHORITY\'}\n\ndef li_spawn_audit(repo,n=SPAWNS_PER_LI_SPECIES):\n    if not G:preflight(repo)\n    base=Path(repo)/\'experimentos/balance_nuevo/individual_variance_v0.1\'\n    if str(base) not in sys.path:sys.path.insert(0,str(base))\n    vi=importlib.import_module(\'individual_variance\')\n    assert vi.CONFIG[\'schema_version\']==\'monster-individual-variance-v0.4\'\n    groups=G[\'suffix\'][\'groups\']; abilities=G[\'suffix\'][\'abilities\']\n    out=[]\n    for sid in sorted(vi.CONFIG[\'species\']):\n        mutants=0; counter={\'specialized\':0,\'exceptional\':0,\'ascended\':0}; suffix_counts=defaultdict(int)\n        for rep in range(n):\n            seed=b.stable_seed(TAG,\'LI_SPAWN\',sid,rep)\n            got=vi.instantiate(sid,seed)\n            spec=vi.CONFIG[\'species\'][sid];qq=got[\'individual_axis_quality\']\n            assert all(0<=q<=1 for q in qq.values())\n            for stat in vi.CONFIG[\'variable_stats\']:\n                assert spec[\'base\'][stat]<=got[\'stats\'][stat]<=spec[\'upper_envelope\'][stat]\n            for key,rule in spec[\'attacks\'].items():assert got[\'attacks\'][key] in rule[\'ladder\']\n            if not got[\'mutant\']:\n                assert got[\'loot_multiplier\']==got[\'xp_multiplier\']==1.0\n                continue\n            mutants+=1\n            assert got[\'loot_multiplier\']==got[\'xp_multiplier\']==1.5\n            assert got[\'individual_power_score\']>=vi.CONFIG[\'mutant\'][\'threshold_by_variable_axis_count\'][str(len(qq))]\n            extreme={ax for ax,q in qq.items() if q>=G[\'suffix\'][\'taxonomy\'][\'extreme_q_threshold\']}\n            label=\'ascended\' if len(extreme)>=3 else (\'exceptional\' if len(extreme)==2 else \'specialized\')\n            counter[label]+=1\n            # Semantic groups derive deterministically from same quality axes; no extra RNG.\n            present={name for name,axes in groups.items() if any(ax in extreme for ax in axes)}\n            if len(extreme)<2:\n                # A specialized Mutante always has a dominant *available* group,\n                # even when no q-axis crosses the exceptional 0.95 cutoff.\n                eligible={name:max(qq[ax] for ax in axes if ax in qq)\n                          for name,axes in groups.items() if any(ax in qq for ax in axes)}\n                assert eligible,(sid,qq)\n                present={max(sorted(eligible),key=lambda name:eligible[name])}\n            assert all(k in abilities for k in present)\n            for g in present:suffix_counts[g]+=1\n        rate=mutants/n\n        # This finite sample is a diagnostic. Strict <1% applies to true spawn process,\n        # not a hard-fail each finite 20k-sample draw.\n        out.append({\'species\':sid,\'spawns\':n,\'mutants\':mutants,\'sample_rate\':rate,\n          **counter,\'detected_suffix_groups\':dict(suffix_counts),\n          \'population_target\':float(vi.CONFIG[\'mutant\'][\'target_incidence\']),\n          \'hard_incidence_contract\':float(vi.CONFIG[\'mutant\'][\'maximum_incidence\'])})\n    return out\n\ndef add_historic(state,mid,mode):\n    if mode==\'FROZEN_T0\':return\n    hist=G[\'six\'][\'species\'][mid][\'variance\'][\'historical_upper_evidence\']\n    s=state.monster\n    if mode in (\'HIST_EVASION\',\'HIST_JOINT_MAX\'):\n        s.evasion=max(s.evasion,float(hist[\'upper_envelope\'][\'evasion\']))\n    if mode in (\'HIST_DEFENSE\',\'HIST_JOINT_MAX\'):\n        s.defense=max(s.defense,float(hist[\'upper_envelope\'][\'defense\']))\n    if mode==\'HIST_JOINT_MAX\':\n        s.hp_max=max(s.hp_max,float(hist[\'upper_envelope\'][\'hp\']));s.hp=s.hp_max\n        s.precision=max(s.precision,float(hist[\'upper_envelope\'][\'precision\']))\n        s.tenacity=max(s.tenacity,float(hist[\'upper_envelope\'][\'tenacity\']))\n    if mode in (\'HIST_ATTACK\',\'HIST_JOINT_MAX\'):\n        prof=copy.deepcopy(state.monster_profile)\n        for axis,dst in [(\'basic_damage\',(\'stats\',\'basic_damage\')),\n                         (\'technique_direct_damage\',(\'technique\',\'params\',\'direct_damage\'))]:\n            if axis not in hist[\'attacks\']:continue\n            ladder=hist[\'attacks\'][axis][\'ladder\']\n            if not ladder:raise RuntimeError(\'EMPTY_HISTORICAL_LADDER:\'+axis)\n            p=prof\n            for v in dst[:-1]:p=p[v]\n            # Select historical last measured ladder candidate, NOT a new formula.\n            p[dst[-1]]=ladder[-1]\n        state.monster_profile=prof\n    assert state.monster.hp_max>=b.G[\'registry\'][\'profiles\'][mid][\'stats\'][\'hp\']\n    assert not state.monster_profile[\'unique\']\n\ndef jobs():\n    work=[]\n    for root in ROOTS:\n      for gear in GEARS:\n       for mid in MONSTERS:\n        for tier in TIERS:\n         for build in BUILDS:\n          for policy in POLICIES:\n           for mode in MODES:\n            work.append(dict(phase=\'HISTORICAL_SENSITIVITY\',root=root,gear=gear,\n              monster=mid,tier=tier,build_id=build,policy=policy,\n              mode=mode,variant=\'CANON\',R=R))\n    # Re-check V04 adjustments in the frozen context and the extreme joint scenario.\n    for root,build,variants in [\n       (\'fuego\',\'OFF_T1_0\',(\'FIRE_DOT12\',\'FIRE_DOT_TICKS2\')),\n       (\'agua\',\'OFF_T1_2\',(\'WATER_OFF_EFF_MINUS2\',)),\n       (\'agua\',\'DEF_T1_2\',(\'WATER_DEF_EFF_MINUS2\',))]:\n      for gear in GEARS:\n       for mid in MONSTERS:\n        for tier in TIERS:\n         for mode in (\'FROZEN_T0\',\'HIST_JOINT_MAX\'):\n          for policy in POLICIES:\n           for variant in variants:\n            work.append(dict(phase=\'HISTORICAL_SENSITIVITY\',root=root,gear=gear,\n                monster=mid,tier=tier,build_id=build,policy=policy,\n                mode=mode,variant=variant,R=R))\n    assert len(work)==(5*4*2*2*8*2*5+ (2+1+1)*4*2*2*2*2)\n    expected=len(work)*R\n    assert expected== (5*4*2*2*8*2*5+4*4*2*2*2*2)*32\n    return work\n\ndef fight(job,rep):\n    root,mid,tier=job[\'root\'],job[\'monster\'],job[\'tier\']\n    eng=b.G[\'eng\']; paths=v3.choice_paths(root,job[\'build_id\'])\n    state=b.make_state(root,paths,job[\'gear\'],mid,tier)\n    add_historic(state,mid,job[\'mode\'])\n    v4.apply_variant(state,job,paths)\n    rng=random.Random(b.stable_seed(TAG,root,job[\'gear\'],mid,tier,rep))\n    recog=b.Recognition();natural=anticipated=due=miss=defs=aoe=basics=0\n    for _ in range(b.MAX_ROUNDS):\n        if not state.player.alive() or not state.monster.alive():break\n        state.round_no+=1;eng.player_turn_start(state,rng)\n        if not state.player.alive():break\n        tid=b.causal_policy_decision(state,job[\'policy\'],defs)\n        c=state.compiled[tid]; prev=state.monster.hp\n        ok=eng.execute_player_technique(state,rng,c)\n        if not ok:eng.execute_basic(state,rng);category=\'PLAYER_BASIC\';basics+=1;isdef=False\n        else:\n            isdef=c[\'role\']==\'DEFENSIVE\';category=\'PLAYER_DEFENSIVE_TECHNIQUE\' if isdef else \'PLAYER_UNITARGET_TECHNIQUE\'\n        defs+=int(isdef);aoe+=int(category==\'PLAYER_AOE_TECHNIQUE\')\n        dealt=max(0.0,prev-state.monster.hp)\n        recognized=recog.observe(category,\'EFECTIVA\' if (isdef or dealt>0) else \'FALLIDA\') if tier==\'T2\' else None\n        if not state.monster.alive():break\n        eng.monster_turn_start(state,rng)\n        if not state.monster.alive():break\n        trigger=b.maybe_arm(state,mid,tier,recognized,dealt)\n        natural+=int(trigger==\'NATURAL\');anticipated+=int(trigger==\'ANTICIPATORY\')\n        params=state.monster_profile[\'technique\'][\'params\']\n        was_due=state.round_no%int(params[\'cadence\'])==0 and not state.monster.skip_next_action\n        eng.execute_monster_turn(state,rng)\n        if was_due:\n            due+=1;miss+=int(not state.monster_recent_actions or state.monster_recent_actions[-1]!=\'TECHNIQUE\')\n        eng.end_round(state)\n    return {\'win\':int(state.monster.hp<=0 and state.player.hp>0),\n       \'hp_pct\':max(0.,state.player.hp)/state.player.hp_max,\'qi_spent\':float(state.metrics.qi_spent),\n       \'qi_final\':max(0.,state.player.qi),\'rounds\':int(state.round_no),\n       \'player_damage_dot\':float(state.metrics.player_damage_dot),\n       \'player_damage_direct\':float(state.metrics.player_damage_direct),\n       \'defense_uses\':defs,\'natural\':natural,\'anticipatory\':anticipated,\n       \'canonical_due\':due,\'canonical_due_missed\':miss,\'fallback_basics\':basics,\'aoe_uses\':aoe,\n       \'timeout\':int(state.round_no>=b.MAX_ROUNDS and state.player.alive() and state.monster.alive())}\n\ndef worker_init(repo):\n    b.load_sources(repo);b.validate_contracts()\n    if len(b.event_tests())!=10:raise RuntimeError(\'EVENT_PARITY_FAILED\')\n    preflight(repo)\n\ndef execute_job(job):\n    return [{**{k:v for k,v in job.items() if k!=\'R\'},\'replicate\':rep,**fight(job,rep)} for rep in range(job[\'R\'])]\n\ndef signature(job):return tuple(job.get(k,\'\') for k in (\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'policy\',\'mode\',\'variant\'))\n\ndef run_jobs(work,out,workers,repo):\n    cp=out/\'checkpoints\'/\'SENSITIVITY\';cp.mkdir(parents=True,exist_ok=True)\n    CHUNK=20;chunks=[work[i:i+CHUNK] for i in range(0,len(work),CHUNK)]\n    progress=b.Progress(len(work)*R,\'V05A HISTORICAL (NO MUTANTS)\');rows=[];cached=0\n    try:\n      with (ProcessPoolExecutor(max_workers=workers,initializer=worker_init,initargs=(str(repo),)) if workers>1 else nullcontext(None)) as pool:\n       for n,ch in enumerate(chunks):\n        dest=cp/f\'chunk_{n:04d}.jsonl.gz\';meta=cp/f\'chunk_{n:04d}.meta.json\'\n        count=sum(x[\'R\'] for x in ch);h=hashlib.sha256(json.dumps(ch,sort_keys=True).encode()).hexdigest()\n        if dest.exists() and meta.exists():\n            if json.loads(meta.read_text())!={\'count\':count,\'job_sha\':h,\'data_sha\':sha(dest)}:\n                raise RuntimeError(\'TAMPERED_CHECKPOINT:\'+str(dest))\n            with gzip.open(dest,\'rt\',encoding=\'utf8\') as f:rec=[json.loads(x) for x in f if x.strip()]\n            if len(rec)!=count or {(signature(x),x[\'replicate\']) for x in rec}!={(signature(y),r) for y in ch for r in range(y[\'R\'])}:\n                raise RuntimeError(\'CHECKPOINT_CONTENT_MISMATCH\')\n            rows.extend(rec);cached+=1;progress.update(count);continue\n        if dest.exists() or meta.exists():raise RuntimeError(\'PARTIAL_CHECKPOINT:\'+str(dest))\n        rec=[x for batch in ([execute_job(j) for j in ch] if pool is None else pool.map(execute_job,ch,chunksize=1)) for x in batch]\n        if len(rec)!=count:raise RuntimeError(\'CHUNK_SIZE_MISMATCH\')\n        progress.cp();temp=dest.with_suffix(dest.suffix+\'.tmp\')\n        with gzip.open(temp,\'wt\',encoding=\'utf8\',compresslevel=5) as f:\n            for x in rec:f.write(json.dumps(x,sort_keys=True,allow_nan=False)+\'\\n\')\n        os.replace(temp,dest);save(meta,{\'count\':count,\'job_sha\':h,\'data_sha\':sha(dest)})\n        rows.extend(rec);progress.resume();progress.update(count)\n      progress.done()\n    except BaseException:progress.err();raise\n    return rows,{\'cached_chunks\':cached,\'total_chunks\':len(chunks),\'fights\':len(rows)}\n\ndef summarize(df,out):\n    d=df.copy();keys=[\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'policy\',\'variant\']\n    # Paired historical resistance vs frozen baseline for identical build/candidate.\n    baseline=d[d[\'mode\']==\'FROZEN_T0\'].set_index(keys+[\'replicate\']).sort_index()\n    outrows=[]\n    for mode in MODES[1:]:\n        treated=d[d[\'mode\']==mode].set_index(keys+[\'replicate\']).sort_index()\n        common=baseline.index.intersection(treated.index)\n        if len(common)==0:raise RuntimeError(\'NO_PAIRED_\'+mode)\n        paired=pd.DataFrame(index=common)\n        paired[\'delta_win\']=(treated.loc[common,\'win\']-baseline.loc[common,\'win\']).to_numpy()\n        paired[\'delta_hp\']=(treated.loc[common,\'hp_pct\']-baseline.loc[common,\'hp_pct\']).to_numpy()\n        paired[\'delta_qi\']=(treated.loc[common,\'qi_spent\']-baseline.loc[common,\'qi_spent\']).to_numpy()\n        for label,grp in paired.groupby(level=keys):\n            diff=grp[\'delta_win\']; n=len(diff);mean=diff.mean()\n            ci=1.96*float(diff.std(ddof=1))/math.sqrt(n) if n>1 else None\n            outrows.append({**dict(zip(keys,label)),\'monster_mode\':mode,\'R\':n,\n                \'delta_win_pp\':float(100*mean),\'ci95_low_pp\':float(100*(mean-ci)) if ci is not None else None,\n                \'ci95_high_pp\':float(100*(mean+ci)) if ci is not None else None,\n                \'delta_hp_pct\':float(grp.delta_hp.mean()),\'delta_qi_spent\':float(grp.delta_qi.mean())})\n    a=pd.DataFrame(outrows);a.to_csv(out/\'PAIRED_SENSITIVITY.csv.gz\',index=False,compression=\'gzip\')\n    g=d.groupby([\'root\',\'monster\',\'tier\',\'mode\',\'build_id\',\'variant\',\'policy\'],as_index=False).agg(\n        R=(\'win\',\'size\'),win_rate=(\'win\',\'mean\'),qi_spent=(\'qi_spent\',\'mean\'),\n        hp_pct=(\'hp_pct\',\'mean\'),dot_damage=(\'player_damage_dot\',\'mean\'),\n        canonical_due_missed=(\'canonical_due_missed\',\'sum\'))\n    g.to_csv(out/\'STRATIFIED_SUMMARY.csv\',index=False)\n    # In arms with same root/build/mode/policy, compare canon vs V04 candidates by pair.\n    candidate=[]\n    idx=[\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'policy\',\'mode\',\'replicate\']\n    original=d[d.variant==\'CANON\'].set_index(idx)\n    for c in (\'FIRE_DOT12\',\'FIRE_DOT_TICKS2\',\'WATER_OFF_EFF_MINUS2\',\'WATER_DEF_EFF_MINUS2\'):\n        q=d[d.variant==c].set_index(idx)\n        if len(q)==0:continue\n        common=original.index.intersection(q.index)\n        if len(common)!=len(q):raise RuntimeError(\'UNPAIRED_VARIANT:\'+c)\n        diff=q.loc[common,\'win\']-original.loc[common,\'win\'];qi=q.loc[common,\'qi_spent\']-original.loc[common,\'qi_spent\']\n        for (root,gear,monster,tier,build,policy,mode),gr in pd.DataFrame({\'dwin\':diff,\'dqi\':qi}).groupby(level=idx[:-1]):\n            candidate.append(dict(variant=c,root=root,gear=gear,monster=monster,tier=tier,build_id=build,policy=policy,mode=mode,\n                R=len(gr),delta_win_pp=100*float(gr.dwin.mean()),delta_qi_spent=float(gr.dqi.mean())))\n    pd.DataFrame(candidate).to_csv(out/\'CANDIDATE_PAIRED.csv\',index=False)\n    return {\'sensitivity_contexts\':len(a),\'variant_contexts\':len(candidate),\n       \'worst_stress_delta_win_pp\':float(a.delta_win_pp.min()),\'max_due_missed\':int(d.canonical_due_missed.sum())}\n\ndef main(argv=None):\n    ap=argparse.ArgumentParser();ap.add_argument(\'--repo\',required=True);ap.add_argument(\'--out\',required=True)\n    ap.add_argument(\'--v05-runner\',required=True);ap.add_argument(\'--v04-runner\',required=True)\n    ap.add_argument(\'--v03-runner\',required=True);ap.add_argument(\'--base-runner\',required=True)\n    ap.add_argument(\'--workers\',type=int,default=2);arg=ap.parse_args(argv)\n    repo=Path(arg.repo).resolve();out=Path(arg.out).resolve();out.mkdir(parents=True,exist_ok=True)\n    if arg.workers not in (1,2):raise RuntimeError(\'WORKERS_MUST_BE_1_OR_2\')\n    source,contracts,events=v3.initial_locks(repo,arg.v03_runner,arg.base_runner)\n    assert source[\'commit\']==PIN and len(events)==10 and all(x==\'PASS\' for x in events.values())\n    auth=preflight(repo);variant_pre=v4.cost_preflight()\n    work=jobs();planned=sum(x[\'R\'] for x in work)\n    sig={\'source_commit\':PIN,\'source\':{k:v[\'sha256\'] for k,v in source[\'files\'].items()},\n       \'extra\':{k:v[\'sha256\'] for k,v in auth[\'authority_sha\'].items()},\n       \'runner_sha\':{name:sha(getattr(arg,name)) for name in [\'base_runner\',\'v03_runner\',\'v04_runner\',\'v05_runner\']},\n       \'seed\':TAG,\'grid_hash\':hashlib.sha256(json.dumps(work,sort_keys=True).encode()).hexdigest()}\n    f=out/\'CHECKPOINT_RUN_SIGNATURE.json\'\n    if f.exists():\n        if json.loads(f.read_text())!=sig:raise RuntimeError(\'CHECKPOINT_SIGNATURE_MISMATCH_USE_NEW_RESULTS\')\n    else:save(f,sig)\n    save(out/\'SOURCE_LOCK.json\',source);save(out/\'AUTHORITY_PREFLIGHT.json\',auth)\n    save(out/\'EVENT_PARITY_TESTS.json\',events);save(out/\'CANDIDATE_COST_PREFLIGHT.json\',variant_pre)\n    spawn=li_spawn_audit(repo)\n    save(out/\'LI_MUTANT_SPAWN_SEMANTICS.json\',spawn)\n    save(out/\'STRESS_DESIGN.json\',{\'modes\':MODES,\'scope\':\'NONCANONICAL_HISTORICAL_AXES_NO_SUFFIX_ABILITIES\',\n      \'represented_fights\':planned,\'R\':R,\'li_spawn_count\':sum(z[\'spawns\'] for z in spawn),\n      \'lii_real_mutant_fights\':0,\'why\':\'LII frozen individual variance collapsed to T0 through T2\'})\n    rows,status=run_jobs(work,out,arg.workers,repo)\n    df=pd.DataFrame(rows);assert len(df)==planned\n    key=[\'root\',\'gear\',\'monster\',\'tier\',\'build_id\',\'policy\',\'mode\',\'variant\',\'replicate\']\n    if df.duplicated(key).any():raise RuntimeError(\'DUPLICATE_ROWS\')\n    if df[[\'win\',\'hp_pct\',\'qi_spent\',\'rounds\']].isna().any().any():raise RuntimeError(\'NAN\')\n    if not np.isfinite(df.select_dtypes(\'number\').to_numpy()).all():raise RuntimeError(\'NONFINITE\')\n    for k in (\'timeout\',\'aoe_uses\',\'canonical_due_missed\'):\n        if df[k].sum()!=0:raise RuntimeError(\'HARD_GUARD:\'+k+\':\'+str(df[k].sum()))\n    df.to_csv(out/\'RAW_SENSITIVITY.csv.gz\',index=False,compression=\'gzip\')\n    metrics=summarize(df,out)\n    save(out/\'CHECKPOINT_HASHES.json\',{str(p.relative_to(out)):sha(p) for p in sorted((out/\'checkpoints\').rglob(\'chunk_*.jsonl.gz\'))})\n    decision={\'status\':\'PASS_STRUCTURAL_REVIEW_REQUIRED_NOT_LII_MUTANT_FREEZE\',\'fights\':planned,\n      \'actual_LII_mutant_fights\':0,\'historical_sensitivity_fights\':planned,\'li_mutant_spawns\':sum(x[\'spawns\'] for x in spawn),\n      \'LII_candidate_changes_canonical\':False,\'LII_concordance_covered\':False,\'legal_multiroot_covered\':False,\n      \'cannot_freeze_full_LII\':True,\'event_tests\':10,\'checkpoint\':status,\'metrics\':metrics,\n      \'next_gates\':[\'HUMAN_RATIFY_LII_INDIVIDUAL_VARIANCE_ENVELOPE\',\'LII_SUFFIX_EVENT_ADAPTER_QA\',\n                    \'LII_MULTIRROOT_AND_CONCORDANCE_LEGALITY_GATE\'],\n      \'note\':\'Historical upper values from separate measured trials are NOT a ratified joint spawn, mutation or suffix.\'}\n    save(out/\'DECISION_GATES.json\',decision)\n    cpmap={\'V05_RUNNER.py\':arg.v05_runner,\'V04_RUNNER.py\':arg.v04_runner,\n      \'V03_RUNNER.py\':arg.v03_runner,\'LAB_RUNNER.py\':arg.base_runner}\n    for name,p in cpmap.items():shutil.copy2(p,out/name)\n    outputs=[\'SOURCE_LOCK.json\',\'AUTHORITY_PREFLIGHT.json\',\'EVENT_PARITY_TESTS.json\',\n      \'CANDIDATE_COST_PREFLIGHT.json\',\'LI_MUTANT_SPAWN_SEMANTICS.json\',\'STRESS_DESIGN.json\',\n      \'CHECKPOINT_RUN_SIGNATURE.json\',\'RAW_SENSITIVITY.csv.gz\',\'PAIRED_SENSITIVITY.csv.gz\',\n      \'STRATIFIED_SUMMARY.csv\',\'CANDIDATE_PAIRED.csv\',\'CHECKPOINT_HASHES.json\',\'DECISION_GATES.json\',*cpmap]\n    info={}\n    for filename in outputs:\n        q=out/filename\n        if not q.exists():raise RuntimeError(\'MISSING_OUTPUT:\'+filename)\n        info[filename]={\'sha256\':sha(q),\'bytes\':q.stat().st_size}\n    save(out/\'PACKAGE_MANIFEST.json\',{\'files\':info,\'lab_id\':ID})\n    dest=out/(ID+\'_REVIEW.zip\')\n    with zipfile.ZipFile(dest,\'w\',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:\n        for x in outputs+[\'PACKAGE_MANIFEST.json\']:z.write(out/x,arcname=x)\n    with zipfile.ZipFile(dest) as z:\n        if z.testzip() is not None:raise RuntimeError(\'ZIP_CRC_FAIL\')\n    print(\'REVIEW\',dest,\'SHA-256\',sha(dest),\'COMBATS\',planned,\'LI SPAWNS\',sum(x[\'spawns\'] for x in spawn))\n    return decision\n\nif __name__==\'__main__\':main()\n'}
RUNNER_SHA256 = {'LAB_RUNNER.py': 'fc14ff185419d694160121ad9684539bfd324ea171fda8fe8ef8261566510fa9', 'LII_TRAMO_I_QI_BREAKPOINT_AND_POLICY_DIAG_V03_RUNNER.py': 'bfcac6ba62b29798ad333bcf9f44534f0d45b5f2948ace6a4f6e356b7b885105', 'LII_V04_EXHAUSTIVE_BALANCE_GATE_RUNNER.py': 'bafc8a14904d2c61276e2feafe7e3e08476db22fdc80f3b2b227aa487384eee0', 'LII_V05_VARIANCE_STRESS_CROSSCHECK_RUNNER.py': 'bc2446c315210c5eaba5106b5a9000fd28a54bf36626a3757873c602c49dee56'}
for filename,content in RUNNER_SOURCES.items():
    target=LAB/filename
    target.write_text(content,encoding='utf8')
    py_compile.compile(str(target),doraise=True)
    got=hashlib.sha256(target.read_bytes()).hexdigest()
    if got!=RUNNER_SHA256[filename]:raise RuntimeError('EMBEDDED_RUNNER_HASH_FAIL:'+filename)
    print(filename,'SHA-256 OK',got[:16])
if str(LAB) not in sys.path:sys.path.insert(0,str(LAB))
for module in ('LII_V05_VARIANCE_STRESS_CROSSCHECK_RUNNER','LII_V04_EXHAUSTIVE_BALANCE_GATE_RUNNER',
               'LII_TRAMO_I_QI_BREAKPOINT_AND_POLICY_DIAG_V03_RUNNER','LAB_RUNNER'):
    sys.modules.pop(module,None)
import LAB_RUNNER as b
import LII_TRAMO_I_QI_BREAKPOINT_AND_POLICY_DIAG_V03_RUNNER as v3
import LII_V04_EXHAUSTIVE_BALANCE_GATE_RUNNER as v4
import LII_V05_VARIANCE_STRESS_CROSSCHECK_RUNNER as v5


## 3. Preflight de autoridad obligatorio

Comprueba los **ocho blobs del puente LII T0/T1/T2**, cuatro documentos/módulos de mutantes y variabilidad, diez paridades de eventos y el coste de los candidatos V04. La corrida **se detiene sin simular** si detecta un cambio real de valores, fuente o contratos.

En la salida verás `lii_mutant_spawn_possible_under_frozen_ordinary_variance: false`: es una **restricción real del freeze**, no un fallo de Colab.

In [ ]:
base_file=LAB/'LAB_RUNNER.py';v03_file=LAB/'LII_TRAMO_I_QI_BREAKPOINT_AND_POLICY_DIAG_V03_RUNNER.py'
v04_file=LAB/'LII_V04_EXHAUSTIVE_BALANCE_GATE_RUNNER.py';v05_file=LAB/'LII_V05_VARIANCE_STRESS_CROSSCHECK_RUNNER.py'
lock,contracts,events=v3.initial_locks(REPO,v03_file,base_file)
assert lock['commit']==SOURCE_COMMIT and len(events)==10 and all(x=='PASS' for x in events.values())
authority=v5.preflight(REPO)
qi=v4.cost_preflight()
jobs=v5.jobs()
print('SOURCE LOCK:',len(lock['files']),'archivos; EXTRA AUTHORITY:',len(authority['authority_sha']))
print('EVENTS:',len(events),'/10 PASS')
print('LII MUTANTE CANÓNICO AUTORIZADO:',authority['lii_mutant_spawn_possible_under_frozen_ordinary_variance'])
print('ESTRÉS:',authority['lii_historical_upper_envelope'])
print('ESTUDIO PREVISTO:',len(jobs),'contextos; combates:',sum(j['R'] for j in jobs))
print('EFICIENCIA AGUA:',json.dumps(qi['water'],ensure_ascii=False,indent=2))


## 4. Ejecutar V05A y retomar checkpoints

Barra de progreso persistente. Los checkpoints de 20 contextos se guardan con sus hashes; al reejecutar esta celda recupera lo ya completado. **No borres `results`.** Si modificás código o fuente, comenzá en otra carpeta; por seguridad se bloquea el uso de checkpoints incompatibles.

Este test no activará habilidades mutantes en los monstruos LII, porque no tenemos un envelope mutante LII ratificado. NO confundir `HIST_JOINT_MAX` con un spawn posible.

In [ ]:
args=['--repo',str(REPO),'--out',str(OUT),'--v05-runner',str(v05_file),
      '--v04-runner',str(v04_file),'--v03-runner',str(v03_file),
      '--base-runner',str(base_file),'--workers',str(WORKERS)]
review=v5.main(args)


## 5. Resultados y auditoría humana

Los cambios de win-rate para los extremos históricos son sensibilidad no canónica, nunca balance de un mutante realmente generado. Compararemos Fuego/Agua entre baseline y estrés sin alterar la autoridad.

In [ ]:
import pandas as pd
print('DECISIÓN DEL LABORATORIO:')
print(json.dumps(json.loads((OUT/'DECISION_GATES.json').read_text(encoding='utf8')),indent=2,ensure_ascii=False))
print('\nCONTRASTES CANDIDATOS:')
display(pd.read_csv(OUT/'CANDIDATE_PAIRED.csv'))
print('\nDISTRIBUCIÓN SUFIJOS LI:')
display(pd.DataFrame(json.loads((OUT/'LI_MUTANT_SPAWN_SEMANTICS.json').read_text(encoding='utf8'))))
print('\nESTRÉS HISTÓRICO (RESUMEN):')
display(pd.read_csv(OUT/'STRATIFIED_SUMMARY.csv').head(40))


## 6. Descargar el ZIP de revisión

Enviame `LII_V05A_MUTANT_AUTHORITY_AND_HISTORIC_ENVELOPE_CROSSCHECK_REVIEW.zip`; con él definiremos si existe un escenario realista suficientemente fundamentado para abrir el **V05B de mutantes LII con habilidades sufijo y Concordancias multielementales**.

In [ ]:
from google.colab import files
reviewzip=OUT/'LII_V05A_MUTANT_AUTHORITY_AND_HISTORIC_ENVELOPE_CROSSCHECK_REVIEW.zip'
assert reviewzip.is_file(),'Completa la celda 4 antes de descargar'
with zipfile.ZipFile(reviewzip) as z:
    assert z.testzip() is None
print('ZIP OK',reviewzip,hashlib.sha256(reviewzip.read_bytes()).hexdigest())
files.download(str(reviewzip))
